# CARE Wind Farm C — Event-Level Early Fault Detection (Event-Bag / V4)

Protocol `CARE_C_EVENT_MIL_V4`. This is a corrected rebuild of the earlier
`CARE_C_NATIVE_EVENT_MSWT_V3` benchmark.

## Read this before running

**This notebook does not guarantee excellent metrics, and no notebook can.** The
achievable ceiling is set by the data: 58 events from 22 turbines, with the label
attached to a prediction frame that averages ~375 h. What the changes below do is
remove the mechanisms by which V3 was demonstrably *losing* signal, and remove the
mechanisms by which it may have been *gaining fake* signal. Whether that produces
strong numbers is an empirical question this notebook answers honestly, including
printing a refusal to claim SOTA when the ranking does not support it.

## What changed from V3, and why

**Objective/metric mismatch (largest effect).** V3 trained window-level BCE but scored
events with top-10% pooling. A window 300 h before a fault was labelled `1` and pushed
toward 1 by the loss, which degrades exactly the direction the top-10% aggregate needs.
V4 trains at the event-bag level:

```
bag_logit = mean(top-k window logits),  k/K = 0.10
L = BCE(bag_logit, y_event) + 0.3 * BCE(window_logits, 0) on normal events only
```

The first term optimises the statistic that is thresholded at test time. The second
exploits the asymmetry V3 ignored: every window in a normal event is a genuine negative,
whereas most windows in an anomaly event are not genuine positives.

**A 10-hour window cannot see a degradation trend.** V4 adds a multi-horizon
baseline-deviation context vector: trailing means of every channel over 1 d / 7 d / 30 d
in the same robust units, through a small MLP branch. Every model in the main comparison
receives it, so the proposed model has no structural advantage; `use_ctx=False` appears
only as ablation rung A4.

**Normalisation artefacts.** When a channel was near-constant in the baseline year, V3's
`scale` fell back to `1.0` and values saturated at ±12. Which channels saturated differed
per event, so those columns acted as a turbine fingerprint. V4 clips at ±8, records the
degeneracy, and drops channels that are degenerate in >20% of events, that behave as
cumulative counters, or whose *clipping rate alone* predicts the label.

**Selection on noise.** V3 early-stopped on validation event PR-AUC over 6 positives,
which takes a handful of discrete values, with `PATIENCE=3` and `MAX_EPOCHS=12` — so most
runs stopped mid-anneal. V4 uses `0.7 * event ROC-AUC + 0.3 * window ROC-AUC` on inner
validation, EMA weights, warmup + full cosine, 40 epochs, patience 8.

**Statistical power.** V3 reported 13 test events (6 positive), where recall is quantised
at 1/6. V4's primary evaluation is 5-fold asset-grouped nested CV over all 58 events
(27 positive), with the threshold taken from each fold's own inner validation. The frozen
32/13/13 split is retained as a secondary result. Confidence intervals use a cluster
bootstrap over **assets**, because events from one turbine are not independent.

**Bugs fixed.** Endpoint underflow crash in the cache; `torch.load` without
`weights_only`; "predict everything positive" being a legal threshold candidate;
DataLoader worker forks of the whole dataset; event-count bootstrap on 4 test assets;
duplicated and truncated figure cells; hardcoded confusion-matrix and lead-time numbers.

## Honesty constraints

- Every choice here — architecture, epochs, threshold, aggregation, hygiene tolerances —
  must be fixed from inner validation folds only. Decide the design, run it once, report
  what it says. The moment you tune against the pooled 58-event numbers, the confidence
  intervals and any SOTA statement become invalid.
- The independent test units are **events** (58 out-of-fold, 27 positive). Window counts
  are repeated, overlapping observations of those events and must never be reported as a
  sample size.
- The named modern architectures are task adaptations under one common CARE protocol, not
  byte-identical reproductions of external repositories.

## Before the first run

Delete or rename any existing `cache_CARE_C_EVENT_MIL_V4` directory. The npz layout
gained a `ctx` key and stale files will raise `KeyError`.


In [ ]:
# CELL 1 — Imports and fixed protocol

import os, gc, json, math, random, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, balanced_accuracy_score,
    accuracy_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

PROTOCOL_VERSION = "CARE_C_EVENT_MIL_V4"

SEEDS = [42, 123, 2026]

LOOKBACK = 60                        # 10 h at 10-min resolution
HEALTHY_STATUS_IDS = {0, 2}          # filtering/calibration only, never a model input

# --- normalisation hygiene -----------------------------------------------------------
CLIP = 8.0                           # was 12: +/-12 saturation was fingerprinting events
DEGEN_TOL = 1e-5
DEGEN_MAX_EVENT_FRAC = 0.20          # drop a channel degenerate in >20% of events
RAMP_TOL = 0.90                      # drop cumulative-counter channels
SAT_AUC_TOL = 0.75                   # drop channels whose clip rate alone predicts the label

# --- multi-horizon baseline-deviation context ----------------------------------------
CTX_HORIZONS = (144, 1008, 4320)     # 1 day, 7 days, 30 days

# --- event-bag (MIL) training --------------------------------------------------------
K_PER_BAG = 32
BAGS_PER_CLASS = 6
TOP_FRACTION = 0.10                  # k/K at train == tail quantile at eval
LAMBDA_NEG = 0.3                     # clean-negative window supervision weight

STEPS_PER_EPOCH = 60
MAX_EPOCHS = 40
WARMUP_EPOCHS = 3
PATIENCE = 8
VAL_STRIDE = 6                       # per-epoch validation subsampling: 1 window / hour
EMA_DECAY = 0.998

LR = 8e-4
WEIGHT_DECAY = 0.03
DROPOUT = 0.25
D_MODEL = 64

AUG_CHAN_DROP = 0.10
AUG_JITTER = 0.05
AUG_TIME_MASK = 8

EVAL_CHUNK = 4096                    # windows per forward pass at eval time

# --- evaluation protocol -------------------------------------------------------------
N_FOLDS = 5
OP_WINDOW = 36                       # rolling 6 h
OP_TAIL_Q = 0.90                     # tail quantile of the rolling risk, not its max

# Set True only when you are certain the checkpoints on disk match these hyperparameters.
RESUME_FROM_CHECKPOINTS = False

# --- runtime budget ------------------------------------------------------------------
# A0 and A3 are architecturally identical to "Plain Transformer" and to the proposed
# model, so by default the ablation reuses those runs instead of retraining them.
REUSE_A0_A3 = True
ABLATION_SEEDS = [42]            # the ladder is a relative comparison; 1 seed is enough

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEV = DEVICE

random.seed(SEEDS[0])
np.random.seed(SEEDS[0])
torch.manual_seed(SEEDS[0])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEEDS[0])

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "DejaVu Serif", "Times New Roman"],
    "font.size": 12, "axes.titlesize": 14, "axes.labelsize": 14,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 9.5,
    "axes.linewidth": 1.0, "lines.linewidth": 2.0,
    "figure.dpi": 130, "savefig.dpi": 600,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

PROPOSED = "MS-WTFormer (Proposed)"

print("=" * 100)
print("CARE WIND FARM C — EVENT-BAG BENCHMARK")
print("=" * 100)
print("Protocol :", PROTOCOL_VERSION)
print("Device   :", DEVICE)
print("Seeds    :", SEEDS)
print("Objective: event-bag MIL, top-%d%% window logits" % int(TOP_FRACTION * 100))

In [ ]:
# CELL 2 — Mount Drive and locate the fully extracted CARE dataset
#
# UNCHANGED from the original notebook. Because PROTOCOL_VERSION changed in CELL 1,
# RESULT / CACHE_DIR / CKPT_DIR now resolve to *_CARE_C_EVENT_MIL_V4 and the previous
# V3 outputs are left intact.

from google.colab import drive

MOUNT = Path("/content/gdrive_care_final")

if not (MOUNT / "MyDrive").exists():
    drive.mount(str(MOUNT))
else:
    print("Google Drive already mounted.")

BASE = MOUNT / "MyDrive" / "Tarun" / "new dataset"
EXTRACT_ROOT = BASE / "CARE_To_Compare"
PROJECT = BASE / "CARE_WindFarmC_Project"

assert EXTRACT_ROOT.exists(), f"Missing extracted folder: {EXTRACT_ROOT}"

wind_c_candidates = [
    p for p in EXTRACT_ROOT.rglob("*")
    if p.is_dir() and p.name.strip().lower() == "wind farm c"
]

assert wind_c_candidates, "Wind Farm C was not found below CARE_To_Compare."
WFC = wind_c_candidates[0]

EVENT_INFO_PATH = WFC / "event_info.csv"
FEATURE_DESCRIPTION_PATH = WFC / "feature_description.csv"
DATASET_DIR = WFC / "datasets"

assert EVENT_INFO_PATH.exists()
assert FEATURE_DESCRIPTION_PATH.exists()
assert DATASET_DIR.exists()

RESULT = PROJECT / f"results_{PROTOCOL_VERSION}"
CACHE_DIR = PROJECT / f"cache_{PROTOCOL_VERSION}"
CKPT_DIR = RESULT / "checkpoints"

for d in [PROJECT, RESULT, CACHE_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Wind Farm C  :", WFC)
print("Result dir   :", RESULT)
print("Cache dir    :", CACHE_DIR)
print("Checkpoints  :", CKPT_DIR)

In [ ]:
# CELL 3 — Verify the exact extracted schema
#
# UNCHANGED from the original notebook. The feature_description preview at the end is
# new: use it to confirm that the channels dropped by CELL 5B are counters/accumulators
# rather than physically meaningful sensors.

def read_auto(path, **kwargs):
    df = pd.read_csv(path, sep=";", **kwargs)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=",", **kwargs)
    return df

raw_event_info = read_auto(EVENT_INFO_PATH)

required_event_cols = {"asset_id", "event_id", "event_label", "event_start", "event_end"}
assert required_event_cols.issubset(raw_event_info.columns)

raw_event_info["event_id"] = pd.to_numeric(raw_event_info["event_id"]).astype(int)
raw_event_info["asset_id"] = raw_event_info["asset_id"].astype(str)
raw_event_info["event_label"] = raw_event_info["event_label"].astype(str).str.lower().str.strip()
raw_event_info["is_anomaly"] = raw_event_info["event_label"].eq("anomaly")
raw_event_info["event_start_dt"] = pd.to_datetime(raw_event_info["event_start"])
raw_event_info["event_end_dt"] = pd.to_datetime(raw_event_info["event_end"])

event_files = sorted(
    [p for p in DATASET_DIR.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

assert len(raw_event_info) == 58, len(raw_event_info)
assert len(event_files) == 58, len(event_files)
assert int(raw_event_info["is_anomaly"].sum()) == 27
assert int((~raw_event_info["is_anomaly"]).sum()) == 31
assert raw_event_info["asset_id"].nunique() == 22
assert {int(p.stem) for p in event_files} == set(raw_event_info["event_id"].tolist())

headers = {int(p.stem): read_auto(p, nrows=0).columns.tolist() for p in event_files}
assert set(len(x) for x in headers.values()) == {957}

common_cols = set(next(iter(headers.values())))
for cols in headers.values():
    common_cols &= set(cols)
assert len(common_cols) == 957

reference_cols = headers[next(iter(headers))]
AVG_FEATURES = [c for c in reference_cols if c.lower().endswith("_avg") and c in common_cols]

NF = len(AVG_FEATURES)
assert NF == 238, f"Expected 238 common Avg features, got {NF}"

for c in ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]:
    assert c in common_cols, f"Missing metadata column {c}"

print("Events             :", len(raw_event_info))
print("Anomaly / normal   :", int(raw_event_info.is_anomaly.sum()), "/",
      int((~raw_event_info.is_anomaly).sum()))
print("Assets             :", raw_event_info.asset_id.nunique())
print("Columns/event CSV  :", len(reference_cols))
print("Common Avg features:", NF)

print("\nFEATURE DESCRIPTION (first 20 rows)")
print(read_auto(FEATURE_DESCRIPTION_PATH).head(20).to_string(index=False))

In [ ]:
# CELL 4 — Frozen asset-disjoint split (SECONDARY) + asset folds (PRIMARY)
#
# The first half is UNCHANGED: the frozen 32/13/13 split is retained, but it is now the
# SECONDARY result. The primary evaluation is the 5-fold asset-grouped CV appended at
# the bottom of this cell, which uses all 58 events (27 positives) instead of 13 (6).

LEGACY_SPLIT = PROJECT / "results_final_6h" / "event_split.csv"
FROZEN_SPLIT = RESULT / "event_split.csv"

def verify_split(df):
    assert set(df["split"].unique()) == {"train", "val", "test"}
    assets = {s: set(df.loc[df.split == s, "asset_id"].astype(str))
              for s in ["train", "val", "test"]}
    assert assets["train"].isdisjoint(assets["val"])
    assert assets["train"].isdisjoint(assets["test"])
    assert assets["val"].isdisjoint(assets["test"])
    for s in ["train", "val", "test"]:
        x = df[df.split == s]
        assert x["is_anomaly"].any(), f"{s} contains no anomaly event"
        assert (~x["is_anomaly"]).any(), f"{s} contains no normal event"
    return assets

_V3_SPLIT = PROJECT / "results_CARE_C_NATIVE_EVENT_MSWT_V3" / "event_split.csv"

if FROZEN_SPLIT.exists():
    saved = pd.read_csv(FROZEN_SPLIT)
    event_info = raw_event_info.merge(
        saved[["event_id", "split"]].drop_duplicates(), on="event_id", how="left")
    split_source = "existing V4 split"
elif _V3_SPLIT.exists():
    saved = pd.read_csv(_V3_SPLIT)
    event_info = raw_event_info.merge(
        saved[["event_id", "split"]].drop_duplicates(), on="event_id", how="left")
    split_source = "V3 split (carried over so the secondary result stays comparable)"
elif LEGACY_SPLIT.exists():
    saved = pd.read_csv(LEGACY_SPLIT)
    event_info = raw_event_info.merge(
        saved[["event_id", "split"]].drop_duplicates(), on="event_id", how="left")
    split_source = "previous asset-disjoint split"
else:
    rng = np.random.default_rng(20260824)
    assets_all = raw_event_info["asset_id"].unique().tolist()
    best = None
    for _ in range(20000):
        perm = rng.permutation(assets_all)
        aset = {"train": set(perm[:14]), "val": set(perm[14:18]), "test": set(perm[18:22])}
        temp = raw_event_info.copy()
        temp["split"] = temp["asset_id"].map(
            lambda a: "train" if a in aset["train"] else ("val" if a in aset["val"] else "test"))
        ok = True
        for s in ["train", "val", "test"]:
            z = temp[temp.split == s]
            if len(z) == 0 or z.is_anomaly.sum() == 0 or (~z.is_anomaly).sum() == 0:
                ok = False
                break
        if not ok:
            continue
        targets = {"train": 32, "val": 13, "test": 13}
        score = 0.0
        for s in ["train", "val", "test"]:
            z = temp[temp.split == s]
            score += abs(len(z) - targets[s])
            score += 4 * abs(z.is_anomaly.mean() - raw_event_info.is_anomaly.mean())
        if best is None or score < best[0]:
            best = (score, temp)
    assert best is not None
    event_info = best[1]
    split_source = "deterministically generated asset-disjoint split"

assert event_info["split"].notna().all()
assets = verify_split(event_info)
event_info.to_csv(FROZEN_SPLIT, index=False)

print("Split source:", split_source)
for s in ["train", "val", "test"]:
    x = event_info[event_info.split == s]
    print(f"{s.upper():5s}: events={len(x):2d} | anomaly={int(x.is_anomaly.sum()):2d} | "
          f"normal={int((~x.is_anomaly).sum()):2d} | assets={x.asset_id.nunique():2d}")
print("\nAsset overlap checks: PASS\n")
print("-" * 80)

def asset_folds(event_frame, n_splits=N_FOLDS, seed=20260824):
    """
    Asset-grouped folds. Assets containing at least one anomaly event and assets
    containing none are dealt out separately so every fold has both classes.
    """
    ag = (event_frame.assign(lab=event_frame["is_anomaly"].astype(int))
                     .groupby("asset_id")
                     .agg(n=("lab", "size"), pos=("lab", "sum"))
                     .reset_index())
    ag["has_pos"] = ag["pos"] > 0
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(n_splits)]
    for grp in (ag[ag.has_pos], ag[~ag.has_pos]):
        for i, a in enumerate(rng.permutation(grp.asset_id.to_numpy())):
            folds[i % n_splits].append(str(a))
    return [set(f) for f in folds]


FOLDS = asset_folds(event_info)

print("ASSET-GROUPED FOLDS (primary evaluation, all 58 events)")
ok = True
for i, f in enumerate(FOLDS):
    sub = event_info[event_info.asset_id.isin(f)]
    n_a, n_n = int(sub.is_anomaly.sum()), int((~sub.is_anomaly).sum())
    ok &= (n_a > 0 and n_n > 0)
    print(f"  fold {i}: assets={len(f):2d} events={len(sub):2d} anomaly={n_a:2d} normal={n_n:2d}")
assert ok, "a fold has only one class - change the seed in asset_folds()"

pd.DataFrame([{"fold": i, "asset_id": a} for i, f in enumerate(FOLDS) for a in sorted(f)]
             ).to_csv(RESULT / "asset_folds.csv", index=False)
print("saved:", RESULT / "asset_folds.csv")

---
### Checkpoint 1

Read the printed fold table above. **Every fold must contain at least one anomaly and one
normal event.** If not, change the `seed` argument to `asset_folds`.


In [ ]:
# CELL 5 — Preprocess/cache each event (hygiene flags + 1d/7d/30d context)

def mask_train(series):
    return series.astype(str).str.lower().str.strip().eq("train").to_numpy()


def mask_prediction(series):
    return series.astype(str).str.lower().str.strip().eq("prediction").to_numpy()


def valid_continuity(times, lookback=LOOKBACK):
    """True where the trailing `lookback` samples contain no gap and no reversal."""
    t = pd.Series(times)
    d = t.diff().dt.total_seconds().to_numpy()

    bad = np.zeros(len(t), dtype=np.int64)
    bad[1:] = ((~np.isfinite(d[1:])) | (d[1:] <= 0) | (d[1:] > 20 * 60)).astype(np.int64)

    cs = np.cumsum(bad)
    good = np.zeros(len(t), dtype=bool)
    if len(t) >= lookback:
        i = np.arange(lookback - 1, len(t))
        good[i] = (cs[i] - cs[i - lookback + 1]) == 0
    return good


def build_cache_v2(erow, path_csv, out_path):
    usecols = ["time_stamp", "train_test", "status_type_id"] + AVG_FEATURES
    df = read_auto(path_csv, usecols=usecols, low_memory=False)

    df["time_stamp"] = pd.to_datetime(df["time_stamp"], errors="coerce")
    df = (df.dropna(subset=["time_stamp"])
            .sort_values("time_stamp")
            .drop_duplicates("time_stamp")
            .reset_index(drop=True))

    X = df[AVG_FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    status = pd.to_numeric(df["status_type_id"], errors="coerce").to_numpy()
    train_m = mask_train(df["train_test"])
    pred_m = mask_prediction(df["train_test"])
    healthy = np.isin(status, list(HEALTHY_STATUS_IDS))

    # unsupervised per-event calibration from historical TRAIN rows only
    baseline_m = train_m & healthy
    if baseline_m.sum() < 1000:
        baseline_m = train_m
    B = X[baseline_m]

    med = np.nan_to_num(np.nanmedian(B, axis=0), nan=0.0).astype(np.float32)
    q25 = np.nanpercentile(B, 25, axis=0)
    q75 = np.nanpercentile(B, 75, axis=0)
    iqr = (q75 - q25).astype(np.float32)
    sd = np.nan_to_num(np.nanstd(B, axis=0), nan=0.0).astype(np.float32)

    # remember WHICH channels had no usable scale instead of silently using 1.0
    degen = (((~np.isfinite(iqr)) | (iqr < DEGEN_TOL)) &
             ((~np.isfinite(sd)) | (sd < DEGEN_TOL)))
    scale = np.where((~np.isfinite(iqr)) | (iqr < DEGEN_TOL), sd, iqr)
    scale = np.where((~np.isfinite(scale)) | (scale < DEGEN_TOL), 1.0, scale).astype(np.float32)

    # normalise the FULL series: the 30-day context needs the historical year
    Z = np.where(np.isfinite(X), X, med)
    Z = np.clip((Z - med) / scale, -CLIP, CLIP).astype(np.float32)

    times = pd.to_datetime(df["time_stamp"])
    tnp = times.to_numpy(dtype="datetime64[ns]")
    ev_start = pd.Timestamp(erow.event_start_dt)
    ev_end = pd.Timestamp(erow.event_end_dt)

    # CARE-native event frame; abnormal-status endpoints excluded
    event_m = (pred_m
               & (tnp >= np.datetime64(ev_start))
               & (tnp < np.datetime64(ev_end))
               & healthy)

    ep_global = np.flatnonzero(event_m & valid_continuity(times))

    # an endpoint needs LOOKBACK-1 real predecessors, otherwise the window slice underflows
    ep_global = ep_global[ep_global >= (LOOKBACK - 1)]
    assert len(ep_global) > 0, f"No valid event-frame windows for event {int(erow.event_id)}"

    # multi-horizon trailing means via float64 prefix sums
    csum = np.zeros((len(Z) + 1, NF), dtype=np.float64)
    csum[1:] = np.cumsum(Z, axis=0, dtype=np.float64)
    parts = []
    for H in CTX_HORIZONS:
        lo = np.maximum(0, ep_global - H + 1)
        cnt = (ep_global + 1 - lo).astype(np.float64)[:, None]
        parts.append(((csum[ep_global + 1] - csum[lo]) / cnt).astype(np.float32))
    ctx = np.concatenate(parts, axis=1)
    del csum

    st = int(ep_global.min()) - LOOKBACK + 1
    en = int(ep_global.max()) + 1
    Xs = Z[st:en]
    ep_local = (ep_global - st).astype(np.int32)
    assert ep_local.min() >= LOOKBACK - 1

    # diagnostics stored per event, consumed globally by CELL 5B
    xe = Xs[ep_local].astype(np.float64)
    t = np.arange(len(xe), dtype=np.float64)
    t = (t - t.mean()) / (t.std() + 1e-12)
    ramp = (((xe - xe.mean(0)) / (xe.std(0) + 1e-12)) * t[:, None]).mean(0).astype(np.float32)
    satrate = (np.abs(Xs[ep_local]) >= 0.98 * CLIP).mean(0).astype(np.float32)

    np.savez_compressed(
        out_path,
        x=Xs,
        ctx=ctx.astype(np.float16),
        endpoints=ep_local,
        times_ns=times.iloc[st:en].astype("int64").to_numpy(dtype=np.int64),
        event_start_ns=np.array([ev_start.value], dtype=np.int64),
        event_end_ns=np.array([ev_end.value], dtype=np.int64),
        degen=degen.astype(np.uint8),
        ramp=ramp,
        satrate=satrate,
    )
    return int(baseline_m.sum())


file_map = {int(p.stem): p for p in event_files}

print("=" * 115)
print("PREPROCESSING / CACHE v2")
print("=" * 115)

for k, erow in event_info.sort_values("event_id").reset_index(drop=True).iterrows():
    eid = int(erow.event_id)
    cache_path = CACHE_DIR / f"event_{eid}.npz"

    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=False)
        if "ctx" not in z.files:
            raise KeyError(
                f"{cache_path.name} is a stale v1 cache with no 'ctx'. "
                f"Delete {CACHE_DIR} and rerun this cell."
            )
        print(f"{k+1:02d}/58 event={eid:3d} split={erow.split:5s} cache=HIT  "
              f"windows={len(z['endpoints']):5d} ctx_dim={z['ctx'].shape[1]}")
        continue

    nb = build_cache_v2(erow, file_map[eid], cache_path)
    z = np.load(cache_path, allow_pickle=False)
    print(f"{k+1:02d}/58 event={eid:3d} split={erow.split:5s} cache=NEW  "
          f"windows={len(z['endpoints']):5d} ctx_dim={z['ctx'].shape[1]} baseline_rows={nb:5d}")

print("\nCached events:", len(list(CACHE_DIR.glob('event_*.npz'))))

In [ ]:
# CELL 5B — Global channel hygiene

def build_hygiene(event_frame, cache_dir):
    dg, rp, st, lb = [], [], [], []
    for _, erow in event_frame.iterrows():
        z = np.load(cache_dir / f"event_{int(erow.event_id)}.npz", allow_pickle=False)
        dg.append(z["degen"]); rp.append(z["ramp"]); st.append(z["satrate"])
        lb.append(int(bool(erow.is_anomaly)))

    degen_frac = np.stack(dg).mean(0)
    ramp_mean = np.abs(np.stack(rp)).mean(0)
    sat_mat = np.stack(st)
    lb = np.array(lb)

    sat_auc = np.full(NF, 0.5)
    for j in range(NF):
        if np.ptp(sat_mat[:, j]) > 0:
            sat_auc[j] = roc_auc_score(lb, sat_mat[:, j])

    d_deg = degen_frac > DEGEN_MAX_EVENT_FRAC
    d_ramp = ramp_mean > RAMP_TOL
    d_sat = np.abs(sat_auc - 0.5) > (SAT_AUC_TOL - 0.5)
    drop = d_deg | d_ramp | d_sat

    report = pd.DataFrame({
        "channel": AVG_FEATURES,
        "degen_frac": degen_frac,
        "ramp_mean_abs": ramp_mean,
        "sat_rate_auc": sat_auc,
        "drop_degenerate": d_deg,
        "drop_counter": d_ramp,
        "drop_sat_leak": d_sat,
        "dropped": drop,
    })

    print("CHANNEL HYGIENE")
    print(f"  degenerate baseline scale in >{DEGEN_MAX_EVENT_FRAC:.0%} of events : "
          f"{int(d_deg.sum())}")
    print(f"  cumulative/counter-like (mean |r(value,t)| > {RAMP_TOL})        : "
          f"{int(d_ramp.sum())}")
    print(f"  clipping rate alone predicts the label (AUC > {SAT_AUC_TOL})     : "
          f"{int(d_sat.sum())}")
    print(f"  KEPT                                                          : "
          f"{int((~drop).sum())} / {NF}")

    if int(d_sat.sum()) > 0:
        print("\n  channels whose clipping rate leaks the label (report this in the paper):")
        for j in np.flatnonzero(d_sat)[:15]:
            print(f"    {AVG_FEATURES[j]:45s} AUC={sat_auc[j]:.3f}")

    return np.flatnonzero(~drop).astype(np.int64), report


KEEP_CH, hygiene_report = build_hygiene(event_info, CACHE_DIR)
hygiene_report.to_csv(RESULT / "channel_hygiene.csv", index=False)

NKEEP = len(KEEP_CH)
CTX_KEEP = np.concatenate([KEEP_CH + h * NF for h in range(len(CTX_HORIZONS))])
NCTX = len(CTX_KEEP)

print(f"\nNKEEP = {NKEEP}   NCTX = {NCTX}")
print("saved:", RESULT / "channel_hygiene.csv")
if NKEEP < 150:
    print("\nWARNING: fewer than 150 channels kept. Inspect channel_hygiene.csv against "
          "feature_description.csv and consider raising RAMP_TOL before continuing.")

---
### Checkpoint 2 — the hygiene report

Read `KEPT n / 238` from the cell above before continuing.

- If `drop_sat_leak` is non-empty, that is direct evidence the old ±12 clipping was
  leaking the label. Say so in the paper.
- If fewer than ~150 channels survive, open `channel_hygiene.csv` and compare the dropped
  channels against `feature_description.csv`. Raise `RAMP_TOL` rather than losing real
  sensors.


In [ ]:
# CELL 6 — GPU-resident event store (replaces WindowDataset / DataLoader)

def build_store(event_frame, cache_dir, keep_ch, ctx_keep, device):
    store = []
    offs = torch.arange(-LOOKBACK + 1, 1, device=device)
    for _, erow in event_frame.sort_values("event_id").reset_index(drop=True).iterrows():
        z = np.load(cache_dir / f"event_{int(erow.event_id)}.npz", allow_pickle=False)
        store.append({
            "event_id": int(erow.event_id),
            "asset_id": str(erow.asset_id),
            "label": int(bool(erow.is_anomaly)),
            "split": str(erow.split),
            "x": torch.from_numpy(np.ascontiguousarray(z["x"][:, keep_ch])
                                  ).to(device, torch.float16),
            "ctx": torch.from_numpy(np.ascontiguousarray(
                z["ctx"].astype(np.float32)[:, ctx_keep])).to(device, torch.float16),
            "ep": torch.from_numpy(z["endpoints"].astype(np.int64)).to(device),
            "offs": offs,
            "times_ns": z["times_ns"],
            "event_start_ns": int(z["event_start_ns"][0]),
            "event_end_ns": int(z["event_end_ns"][0]),
        })
    return store


# estimate footprint, fall back to CPU-resident storage if the GPU is small
_est_mb = 0.0
for _, _e in event_info.iterrows():
    _z = np.load(CACHE_DIR / f"event_{int(_e.event_id)}.npz", allow_pickle=False)
    _est_mb += (_z["x"].shape[0] * NKEEP + _z["ctx"].shape[0] * NCTX) * 2 / 1e6

STORE_DEVICE = DEV
if torch.cuda.is_available():
    free_mb = torch.cuda.mem_get_info()[0] / 1e6
    if _est_mb > 0.35 * free_mb:
        STORE_DEVICE = torch.device("cpu")
        print(f"store {_est_mb:.0f} MB vs {free_mb:.0f} MB free -> keeping data on CPU")

STORE = build_store(event_info, CACHE_DIR, KEEP_CH, CTX_KEEP, STORE_DEVICE)

ASSET_OF = np.array([s["asset_id"] for s in STORE])
LABEL_OF = np.array([s["label"] for s in STORE])
EVENTID_OF = np.array([s["event_id"] for s in STORE])

print(f"{len(STORE)} events resident on {STORE_DEVICE} ({_est_mb:.0f} MB)")
print(f"input per window: {LOOKBACK} x {NKEEP}   context: {NCTX}")
for sp in ["train", "val", "test"]:
    idx = [i for i, s in enumerate(STORE) if s["split"] == sp]
    w = sum(len(STORE[i]["ep"]) for i in idx)
    print(f"  {sp.upper():5s}: events={len(idx):2d} windows={w:6d} "
          f"anomaly={int(LABEL_OF[idx].sum()):2d} normal={int((LABEL_OF[idx] == 0).sum()):2d}")


def gather_windows(rec, sel):
    """
    sel : LongTensor of POSITIONS into rec['ep'] (any device)
    ->  (n, LOOKBACK, NKEEP) and (n, NCTX), both float32 on DEV
    """
    sel = sel.to(rec["x"].device)
    rows = rec["ep"][sel][:, None] + rec["offs"][None, :]
    x = rec["x"][rows].float()
    c = rec["ctx"][sel].float()
    if x.device != DEV:
        x = x.to(DEV, non_blocking=True)
        c = c.to(DEV, non_blocking=True)
    return x, c

In [ ]:
# CELL 6D — DIAGNOSTICS (run ONCE, before any training)
#
# D3 asks whether trivial event properties leak the label.
# D5 gives a classical baseline the deep models must beat, on the SAME folds.
# D6 is the leakage control: with uninformative labels, OOF AUC must collapse to ~0.5.
#
# Set RUN_DIAGNOSTICS = False to skip on later runs.

RUN_DIAGNOSTICS = True

if RUN_DIAGNOSTICS:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.ensemble import HistGradientBoostingClassifier

    lab = LABEL_OF
    ast_ = ASSET_OF

    # ---- D3  trivial shortcut baselines ---------------------------------------------
    print("=" * 100)
    print("D3  TRIVIAL SHORTCUT BASELINES")
    print("=" * 100)
    dur_h = np.array([(s["event_end_ns"] - s["event_start_ns"]) / 3.6e12 for s in STORE])
    n_win = np.array([len(s["ep"]) for s in STORE], dtype=float)
    d3 = {}
    for nm, v in [("event duration (h)", dur_h), ("n eligible windows", n_win)]:
        d3[nm] = float(roc_auc_score(lab, v))
        print(f"  {nm:22s} ROC-AUC over all {len(lab)} events = {d3[nm]:.3f}")
    if max(abs(v - 0.5) for v in d3.values()) > 0.15:
        print("  >0.65 or <0.35: a trivial event property carries label information.")
        print("  Report the control and keep the tail-quantile operational statistic.")
    else:
        print("  Both are close to 0.5, so neither event duration nor window count carries")
        print("  usable label information. The length-bias concern is retired empirically;")
        print("  the tail-quantile operational statistic is now prudence, not a fix.")

    # ---- shared fold machinery -------------------------------------------------------
    # Use the SAME asset folds as the deep models, and the same train/val/test roles, so
    # the classical baseline is a like-for-like comparison rather than a looser one.
    def fold_indices(folds):
        out = []
        for fi in range(len(folds)):
            te_a, va_a = folds[fi], folds[(fi + 1) % len(folds)]
            te = np.array([i for i in range(len(STORE)) if ast_[i] in te_a])
            tr = np.array([i for i in range(len(STORE))
                           if ast_[i] not in (te_a | va_a)])
            out.append((tr, te))
        return out

    FOLD_IDX = fold_indices(FOLDS)
    print(f"\n  fold sizes (train/test), val fold excluded from train to match the deep "
          f"models:")
    for fi, (tr, te) in enumerate(FOLD_IDX):
        print(f"    fold {fi}: train={len(tr):2d} (pos={int(lab[tr].sum()):2d})  "
              f"test={len(te):2d} (pos={int(lab[te].sum()):2d})")

    def oof_predict(model, X, y, fold_idx):
        """Returns out-of-fold scores, or None if any fold is unusable."""
        oof = np.full(len(y), np.nan)
        for tr, te in fold_idx:
            if len(np.unique(y[tr])) < 2:
                return None
            model.fit(X[tr], y[tr])
            oof[te] = model.predict_proba(X[te])[:, 1]
        m = ~np.isnan(oof)
        if len(np.unique(y[m])) < 2:
            return None
        return oof

    # ---- D5  classical event-summary baseline ----------------------------------------
    print("\n" + "=" * 100)
    print("D5  CLASSICAL EVENT-SUMMARY BASELINE (same asset folds as the deep models)")
    print("=" * 100)

    def event_summary(rec, roll=36):
        x = rec["x"][rec["ep"]].float().cpu().numpy()
        if len(x) >= roll:
            k = np.ones(roll, dtype=np.float32) / roll
            sm = np.apply_along_axis(lambda c: np.convolve(c, k, mode="valid"), 0, x)
        else:
            sm = x
        return np.concatenate([sm.mean(0), np.quantile(sm, 0.95, axis=0),
                               np.quantile(sm, 0.05, axis=0), np.abs(sm).max(0)]
                              ).astype(np.float32)

    Xev = np.nan_to_num(np.stack([event_summary(r) for r in STORE]),
                        nan=0.0, posinf=0.0, neginf=0.0)
    print("  event feature matrix:", Xev.shape)

    MODELS = {
        "L2 logistic (C=0.003)": make_pipeline(
            StandardScaler(), LogisticRegression(C=0.003, max_iter=5000,
                                                 class_weight="balanced")),
        "L2 logistic (C=0.01)": make_pipeline(
            StandardScaler(), LogisticRegression(C=0.01, max_iter=5000,
                                                 class_weight="balanced")),
        "L2 logistic (C=0.1)": make_pipeline(
            StandardScaler(), LogisticRegression(C=0.1, max_iter=5000,
                                                 class_weight="balanced")),
        "HistGB (depth 2)": HistGradientBoostingClassifier(
            max_depth=2, max_iter=200, learning_rate=0.05, random_state=0),
    }

    d5_oof, d5_scores = {}, {}
    for nm, mdl in MODELS.items():
        oof = oof_predict(mdl, Xev, lab, FOLD_IDX)
        if oof is None:
            print(f"  {nm:22s} SKIPPED (unusable fold)")
            continue
        d5_oof[nm] = oof
        d5_scores[nm] = (roc_auc_score(lab, oof), average_precision_score(lab, oof))
        print(f"  {nm:22s} OOF ROC-AUC={d5_scores[nm][0]:.3f}  "
              f"OOF PR-AUC={d5_scores[nm][1]:.3f}")

    BEST_CLASSICAL = max(d5_scores, key=lambda k: d5_scores[k][0])
    BAR_ROC, BAR_PR = d5_scores[BEST_CLASSICAL]
    prevalence = float(lab.mean())

    pd.DataFrame({"event_id": EVENTID_OF, "asset_id": ast_, "label": lab,
                  **{f"oof_{k}": v for k, v in d5_oof.items()}}
                 ).to_csv(RESULT / "D5_classical_baseline_oof.csv", index=False)

    print(f"\n  event prevalence           : {prevalence:.3f}")
    print(f"  BAR TO BEAT ({BEST_CLASSICAL}): ROC-AUC {BAR_ROC:.3f} | PR-AUC {BAR_PR:.3f}")
    print("  This is the number the transformers must beat to justify themselves, and it")
    print("  goes in the paper as a baseline row regardless of the outcome.")
    if d5_scores.get("HistGB (depth 2)", (1, 1))[0] < BAR_ROC - 0.05:
        print("  Note: the heavily regularised linear model beats gradient boosting, which")
        print("  is the signature of a very small effective sample size. Expect the deep")
        print("  models to need at least the current DROPOUT/WEIGHT_DECAY, possibly more.")
    print("  saved:", RESULT / "D5_classical_baseline_oof.csv")

    # ---- D6  leakage control ---------------------------------------------------------
    print("\n" + "=" * 100)
    print("D6  LABEL PERMUTATION CONTROL (leakage check)")
    print("=" * 100)

    # Asset label composition, printed because it determines which nulls are feasible.
    comp = pd.DataFrame([{"asset_id": a, "n": int((ast_ == a).sum()),
                          "pos": int(lab[ast_ == a].sum())} for a in np.unique(ast_)])
    print(f"  {len(comp)} assets | all-normal={int((comp.pos == 0).sum())} "
          f"all-anomaly={int((comp.pos == comp.n).sum())} "
          f"mixed={int(((comp.pos > 0) & (comp.pos < comp.n)).sum())}")
    print(f"  events per asset: min={comp.n.min()} median={int(comp.n.median())} "
          f"max={comp.n.max()}")

    mdl = MODELS[BEST_CLASSICAL]
    rng = np.random.default_rng(7)
    N_PERM = 50

    # Null A - global label permutation. Preserves the positive count exactly and cannot
    # produce a single-class training fold at this prevalence. This is the primary null.
    a_vals, a_skip = [], 0
    for _ in range(N_PERM):
        yp = rng.permutation(lab)
        oof = oof_predict(mdl, Xev, yp, FOLD_IDX)
        if oof is None:
            a_skip += 1
            continue
        a_vals.append(roc_auc_score(yp, oof))
    a_vals = np.array(a_vals)

    # Null B - asset-block permutation. Reassigns whole assets' label vectors between
    # assets of equal event count, so group structure AND the positive count survive.
    sizes = {}
    for a in np.unique(ast_):
        sizes.setdefault(int((ast_ == a).sum()), []).append(a)
    blocks = {a: lab[ast_ == a].copy() for a in np.unique(ast_)}
    permutable = sum(len(v) for v in sizes.values() if len(v) >= 2)

    b_vals, b_skip = [], 0
    if permutable >= 4:
        for _ in range(N_PERM):
            yp = np.empty(len(lab), int)
            for sz, members in sizes.items():
                dst = rng.permutation(members) if len(members) >= 2 else members
                for src, d in zip(members, dst):
                    yp[ast_ == d] = blocks[src]
            oof = oof_predict(mdl, Xev, yp, FOLD_IDX)
            if oof is None:
                b_skip += 1
                continue
            b_vals.append(roc_auc_score(yp, oof))
    b_vals = np.array(b_vals)

    print(f"\n  model under test: {BEST_CLASSICAL}")
    print(f"  real-label OOF ROC-AUC                    : {BAR_ROC:.3f}")
    if len(a_vals):
        print(f"  null A, global permutation   n={len(a_vals):2d} skipped={a_skip:2d} : "
              f"mean={a_vals.mean():.3f} p95={np.quantile(a_vals, .95):.3f} "
              f"max={a_vals.max():.3f}")
    if len(b_vals):
        print(f"  null B, asset-block swap     n={len(b_vals):2d} skipped={b_skip:2d} : "
              f"mean={b_vals.mean():.3f} p95={np.quantile(b_vals, .95):.3f} "
              f"max={b_vals.max():.3f}")
    else:
        print(f"  null B, asset-block swap                  : not feasible "
              f"({permutable} assets sit in size groups of >=2)")

    ref = a_vals if len(a_vals) else b_vals
    if len(ref):
        p_emp = float((ref >= BAR_ROC).mean())
        print(f"\n  empirical p(null >= real) = {p_emp:.3f}   "
              f"(n={len(ref)} permutations)")
        if ref.mean() > 0.60:
            print("  FAIL: the null mean is well above chance. Something leaks - revisit the")
            print("  CELL 5B hygiene report before trusting any downstream result.")
        elif BAR_ROC <= np.quantile(ref, 0.95):
            print("  CAUTION: the real score does not clear the null p95. The classical")
            print("  baseline's advantage over chance is not established at n=58; treat any")
            print("  small deep-model margin over it with the same scepticism.")
        else:
            print("  PASS: null collapses to chance and the real score clears its p95.")
    print("\n  Note: D6 permutes labels only. It cannot detect a shortcut that is genuinely")
    print("  correlated with the true label, which is what D3 and the CELL 5B clip-rate")
    print("  test are for.")
else:
    print("diagnostics skipped (RUN_DIAGNOSTICS = False)")

---
### Checkpoint 3 — diagnostics decide whether the rewrite was needed

D5 is the informative one. If the classical event-summary baseline reaches out-of-fold
ROC-AUC ≈ 0.8 while the transformers land at 0.6–0.7, the deep pipeline was the limiter
rather than the data — and that baseline belongs in the paper either way.

D6 must collapse to ≈ 0.5. If permuted labels still score well above chance, there is
leakage left in the features.


In [ ]:
# CELL 8 — Event scoring, threshold selection, metrics

@torch.no_grad()
def score_events(model, idxs, chunk=EVAL_CHUNK, return_windows=False, stride=1):
    """
    Event score = mean of the top TOP_FRACTION window LOGITS - identical to the
    statistic optimised by the bag loss in CELL 11.
    stride > 1 subsamples endpoints (use for the per-epoch criterion only).
    """
    model.eval()
    rows, win = [], []
    for i in idxs:
        r = STORE[i]
        pos = torch.arange(0, len(r["ep"]), stride)
        n = len(pos)
        out = torch.empty(n, device=DEV)
        for s in range(0, n, chunk):
            sel = pos[s:s + chunk]
            xb, cb = gather_windows(r, sel)
            out[s:s + len(sel)] = model(xb, cb).float()
        k = max(1, int(math.ceil(TOP_FRACTION * n)))
        rows.append({"event_id": r["event_id"], "asset_id": r["asset_id"],
                     "label": r["label"], "score": float(out.topk(k).values.mean()),
                     "n_windows": n})
        if return_windows:
            ep_np = r["ep"].cpu().numpy()[pos.numpy()]
            win.append(pd.DataFrame({
                "event_id": r["event_id"], "asset_id": r["asset_id"],
                "label": r["label"],
                "timestamp_ns": r["times_ns"][ep_np],
                "event_end_ns": r["event_end_ns"],
                "logit": out.cpu().numpy(),
                "prob": torch.sigmoid(out).cpu().numpy()}))
    ev = pd.DataFrame(rows)
    return (ev, pd.concat(win, ignore_index=True)) if return_windows else ev


def choose_threshold(y, s):
    """
    Interior midpoints only, so 'predict everything positive' is not a legal candidate.
    Objective is MCC-led: MCC uses all four confusion cells and is far less degenerate
    than F1 on a small event set.
    """
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    u = np.unique(s)
    cand = (u[:-1] + u[1:]) / 2.0 if len(u) > 1 else u
    if len(cand) == 0:
        return float(s.mean())
    best = (-2.0, float(cand[0]))
    for t in cand:
        p = (s >= t).astype(int)
        v = 0.7 * matthews_corrcoef(y, p) + 0.3 * f1_score(y, p, zero_division=0)
        if v > best[0]:
            best = (v, float(t))
    return best[1]


def metrics_from_event_df(df, tau=None, pred_col=None):
    y = df.label.to_numpy(int)
    s = df.score.to_numpy(float)
    p = df[pred_col].to_numpy(int) if pred_col else (s >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "PR-AUC": float(average_precision_score(y, s)),
        "ROC-AUC": float(roc_auc_score(y, s)),
        "Precision": float(precision_score(y, p, zero_division=0)),
        "Recall": float(recall_score(y, p, zero_division=0)),
        "F1": float(f1_score(y, p, zero_division=0)),
        "MCC": float(matthews_corrcoef(y, p)),
        "Balanced Accuracy": float(balanced_accuracy_score(y, p)),
        "Accuracy": float(accuracy_score(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Threshold": float(tau) if tau is not None else np.nan,
    }


print("Event-level scoring utilities ready (logit-space top-%d%% aggregation)."
      % int(TOP_FRACTION * 100))

In [ ]:
# CELL 9 — Backbones, shared context head, EMA

class AttentivePool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))

    def forward(self, z):
        a = torch.softmax(self.score(z).squeeze(-1), dim=1)
        return torch.sum(z * a.unsqueeze(-1), dim=1)


class MultiScaleFront(nn.Module):
    """Multi-scale temporal convolution, plus an optional fixed one-level Haar branch."""

    def __init__(self, nf, d, with_wavelet):
        super().__init__()
        self.with_wavelet = with_wavelet
        b = d // 3
        self.c3 = nn.Conv1d(nf, b, 3, padding=1)
        self.c9 = nn.Conv1d(nf, b, 9, padding=4)
        self.c27 = nn.Conv1d(nf, d - 2 * b, 27, padding=13)
        self.res = nn.Linear(nf, d)
        if with_wavelet:
            self.wave = nn.Linear(2 * nf, d)
        self.norm = nn.LayerNorm(d)

    def forward(self, x):
        xc = x.transpose(1, 2)
        ms = torch.cat([F.gelu(self.c3(xc)),
                        F.gelu(self.c9(xc)),
                        F.gelu(self.c27(xc))], dim=1).transpose(1, 2)
        z = ms + self.res(x)
        if self.with_wavelet:
            s = math.sqrt(2.0)
            low = F.pad((x[:, 1:, :] + x[:, :-1, :]) / s, (0, 0, 1, 0))
            high = F.pad((x[:, 1:, :] - x[:, :-1, :]) / s, (0, 0, 1, 0))
            z = z + self.wave(torch.cat([low, high], dim=-1))
        return self.norm(z)


def _encoder(d, h, layers, drop):
    layer = nn.TransformerEncoderLayer(d_model=d, nhead=h, dim_feedforward=2 * d,
                                       dropout=drop, batch_first=True,
                                       activation="gelu", norm_first=True)
    return nn.TransformerEncoder(layer, layers)


class TransformerBB(nn.Module):
    """
    One class covering the whole ablation ladder:
      multiscale=False, wavelet=False, attn_pool=False  -> Plain Transformer
      multiscale=True                                   -> + Multi-scale Temporal
      multiscale=True, wavelet=True                     -> + Haar Wavelet
      multiscale=True, wavelet=True, attn_pool=True     -> MS-WTFormer
    Returns a (B, d) embedding; the head lives in EventModel.
    """

    def __init__(self, nf, d=D_MODEL, h=4, layers=2, drop=DROPOUT,
                 multiscale=False, wavelet=False, attn_pool=False):
        super().__init__()
        self.d = d
        self.front = (MultiScaleFront(nf, d, wavelet) if multiscale
                      else nn.Linear(nf, d))
        self.pos = nn.Parameter(torch.zeros(1, LOOKBACK, d))
        self.enc = _encoder(d, h, layers, drop)
        self.pool = AttentivePool(d) if attn_pool else None

    def forward(self, x):
        z = self.enc(self.front(x) + self.pos)
        return self.pool(z) if self.pool is not None else z.mean(1)


class EventModel(nn.Module):
    """
    backbone -> (+ multi-horizon context branch) -> classification head.
    use_ctx=False reproduces the 10-hours-only input, used as ablation rung A4.
    """

    def __init__(self, backbone, nctx, drop=DROPOUT, use_ctx=True):
        super().__init__()
        self.bb = backbone
        d = backbone.d
        self.use_ctx = use_ctx
        if use_ctx:
            self.ctx = nn.Sequential(nn.LayerNorm(nctx), nn.Dropout(drop),
                                     nn.Linear(nctx, d), nn.GELU(), nn.Linear(d, d))
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 32), nn.GELU(),
                                  nn.Dropout(drop), nn.Linear(32, 1))

    def forward(self, x, c=None):
        z = self.bb(x)
        if self.use_ctx and c is not None:
            z = z + self.ctx(c)
        return self.head(z).squeeze(-1)


class EMA:
    """Weight averaging - removes most of the checkpoint-selection variance."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone().float()

    def state(self, model):
        return {k: self.shadow[k].to(v.dtype) for k, v in model.state_dict().items()}


print("Backbones, context head and EMA ready.")

In [ ]:
# CELL 10 — Modern time-series baseline backbones

class iTransformerBB(nn.Module):
    """Variables are tokens; each token embeds the complete 10-hour history."""

    def __init__(self, nf, d=D_MODEL, h=4, layers=2, drop=DROPOUT):
        super().__init__()
        self.d = d
        self.embed = nn.Sequential(nn.Linear(LOOKBACK, d), nn.GELU(), nn.LayerNorm(d))
        self.enc = _encoder(d, h, layers, drop)
        self.gate = nn.Linear(d, 1)

    def forward(self, x):
        z = self.enc(self.embed(x.transpose(1, 2)))
        a = torch.softmax(self.gate(z).squeeze(-1), dim=1)
        return torch.sum(z * a.unsqueeze(-1), dim=1)


class ModernTCNBlock(nn.Module):
    def __init__(self, d, k, drop):
        super().__init__()
        self.dw = nn.Conv1d(d, d, k, padding=k // 2, groups=d)
        self.pw = nn.Sequential(nn.Conv1d(d, 2 * d, 1), nn.GELU(),
                                nn.Dropout(drop), nn.Conv1d(2 * d, d, 1))
        self.norm = nn.BatchNorm1d(d)

    def forward(self, x):
        return self.norm(x + self.pw(self.dw(x)))


class ModernTCNBB(nn.Module):
    def __init__(self, nf, d=D_MODEL, drop=DROPOUT):
        super().__init__()
        self.d = d
        self.stem = nn.Conv1d(nf, d, 1)
        self.blocks = nn.Sequential(*[ModernTCNBlock(d, k, drop) for k in (7, 15, 31, 15)])

    def forward(self, x):
        return self.blocks(self.stem(x.transpose(1, 2))).mean(-1)


class ScaleMixer(nn.Module):
    def __init__(self, nf, d, drop):
        super().__init__()
        self.proj = nn.Conv1d(nf, d, 3, padding=1)
        self.mix = nn.Sequential(nn.Conv1d(d, d, 5, padding=2, groups=d), nn.GELU(),
                                 nn.Conv1d(d, d, 1), nn.GELU(), nn.Dropout(drop))
        self.norm = nn.BatchNorm1d(d)

    def forward(self, x):
        z = self.proj(x)
        return self.norm(z + self.mix(z)).mean(-1)


class TimeMixerPPBB(nn.Module):
    """Compact TimeMixer++-style multi-resolution adaptation."""

    def __init__(self, nf, d=D_MODEL, drop=DROPOUT):
        super().__init__()
        self.d = d
        self.s = nn.ModuleList([ScaleMixer(nf, d, drop) for _ in range(3)])
        self.gate = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))

    def forward(self, x):
        x = x.transpose(1, 2)
        z = torch.stack([self.s[0](x),
                         self.s[1](F.avg_pool1d(x, 2, 2)),
                         self.s[2](F.avg_pool1d(x, 4, 4))], dim=1)
        a = torch.softmax(self.gate(z).squeeze(-1), dim=1)
        return torch.sum(z * a.unsqueeze(-1), dim=1)


class NonstationaryBB(nn.Module):
    """Series stationarisation + statistics-conditioned Transformer adaptation."""

    def __init__(self, nf, d=D_MODEL, h=4, layers=2, drop=DROPOUT):
        super().__init__()
        self.d = d
        self.inp = nn.Linear(nf, d)
        self.pos = nn.Parameter(torch.zeros(1, LOOKBACK, d))
        self.enc = _encoder(d, h, layers, drop)
        self.stats = nn.Sequential(nn.Linear(2 * nf, d), nn.GELU(), nn.Linear(d, d))

    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        sd = x.std(1, keepdim=True).clamp_min(1e-4)
        z = self.enc(self.inp((x - mu) / sd) + self.pos).mean(1)
        stat = torch.cat([mu.squeeze(1), torch.log(sd.squeeze(1) + 1e-6)], dim=-1)
        return z + self.stats(stat)


print("Modern baseline backbones ready.")

In [ ]:
# CELL 11 — Event-bag (MIL) training

def topk_bag(logits, k):
    """logits: (nbags, K) -> (nbags,). Differentiable through the selected entries."""
    k = max(1, min(k, logits.shape[1]))
    return logits.topk(k, dim=1).values.mean(1)


def augment(x):
    if AUG_CHAN_DROP > 0:
        m = (torch.rand(x.shape[0], 1, x.shape[2], device=x.device) > AUG_CHAN_DROP).float()
        x = x * m
    if AUG_JITTER > 0:
        x = x + AUG_JITTER * torch.randn_like(x)
    if AUG_TIME_MASK > 0 and LOOKBACK > AUG_TIME_MASK:
        s = int(torch.randint(0, LOOKBACK - AUG_TIME_MASK, (1,)).item())
        x[:, s:s + AUG_TIME_MASK] = 0.0
    return x


def train_bag_model(model_fn, train_idx, val_idx, seed, ckpt=None, log=True):
    """
    Event-bag multiple-instance training.

        bag_logit_e = mean of the top-k window logits of event e      (k/K = TOP_FRACTION)
        L = BCE(bag_logit_e, y_e)                                     all events
          + LAMBDA_NEG * BCE(window_logits, 0)                        normal events only

    The first term optimises exactly the statistic thresholded at test time. The second
    exploits the fact that every window in a normal event is a genuine negative, whereas
    a window 300 h before a fault is not a genuine positive.
    """
    if ckpt is not None and RESUME_FROM_CHECKPOINTS and Path(ckpt).exists():
        m = model_fn().to(DEV)
        m.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
        if log:
            print(f"    seed={seed}: loaded checkpoint {Path(ckpt).name}")
        return m, pd.DataFrame()

    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = model_fn().to(DEV)
    eval_model = model_fn().to(DEV)          # reused holder for the EMA weights
    ema = EMA(model)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total = MAX_EPOCHS * STEPS_PER_EPOCH
    warm = WARMUP_EPOCHS * STEPS_PER_EPOCH
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    bce = nn.BCEWithLogitsLoss()

    pos = [i for i in train_idx if STORE[i]["label"] == 1]
    neg = [i for i in train_idx if STORE[i]["label"] == 0]
    assert pos and neg, "training split must contain both classes"
    rng = np.random.default_rng(seed)
    kbag = max(1, int(round(TOP_FRACTION * K_PER_BAG)))

    best, best_state, wait, hist = -np.inf, None, 0, []

    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        run = 0.0
        for _ in range(STEPS_PER_EPOCH):
            chosen = (list(rng.choice(pos, min(BAGS_PER_CLASS, len(pos)), replace=False)) +
                      list(rng.choice(neg, min(BAGS_PER_CLASS, len(neg)), replace=False)))
            xs, cs, ys = [], [], []
            for ridx in chosen:
                r = STORE[ridx]
                sel = torch.from_numpy(rng.integers(0, len(r["ep"]), K_PER_BAG))
                xb, cb = gather_windows(r, sel)
                xs.append(xb); cs.append(cb); ys.append(float(r["label"]))

            x = augment(torch.cat(xs, 0))
            c = torch.cat(cs, 0)
            y = torch.tensor(ys, device=DEV)

            logit_w = model(x, c).view(len(chosen), K_PER_BAG)

            loss = bce(topk_bag(logit_w, kbag), y)

            nm = y < 0.5
            if nm.any():
                lw = logit_w[nm].reshape(-1)
                loss = loss + LAMBDA_NEG * bce(lw, torch.zeros_like(lw))

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); ema.update(model)
            run += float(loss.item())

        # selection criterion: inner validation only, smoothed with window-level ROC
        eval_model.load_state_dict(ema.state(model))
        ve, vw = score_events(eval_model, val_idx, return_windows=True, stride=VAL_STRIDE)
        ev_roc = roc_auc_score(ve.label, ve.score) if ve.label.nunique() == 2 else 0.5
        ev_pr = float(average_precision_score(ve.label, ve.score))
        wi_roc = roc_auc_score(vw.label, vw.logit) if vw.label.nunique() == 2 else 0.5
        crit = 0.7 * ev_roc + 0.3 * wi_roc

        hist.append({"epoch": ep, "loss": run / STEPS_PER_EPOCH,
                     "val_event_roc": ev_roc, "val_event_pr": ev_pr,
                     "val_window_roc": wi_roc, "criterion": crit,
                     "lr": opt.param_groups[0]["lr"]})
        if log:
            print(f"    ep={ep:02d} loss={hist[-1]['loss']:.4f} "
                  f"val_ev_ROC={ev_roc:.3f} val_ev_PR={ev_pr:.3f} "
                  f"val_win_ROC={wi_roc:.3f} crit={crit:.4f}")

        if crit > best + 1e-6:
            best, wait = crit, 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema.state(model).items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                if log:
                    print(f"    early stop at epoch {ep} (best crit={best:.4f})")
                break

    assert best_state is not None
    out = model_fn().to(DEV)
    out.load_state_dict({k: v.to(DEV) for k, v in best_state.items()})
    if ckpt is not None:
        torch.save(best_state, ckpt)
    return out, pd.DataFrame(hist)


print("Event-bag MIL trainer ready.")

In [ ]:
# CELL 12 — Model registry

def safe_name(n):
    return (n.replace(" ", "_").replace("/", "_").replace("+", "plus")
             .replace("(", "").replace(")", "").replace("-", "_")
             .replace("\u2014", "-"))


def make_registry(nf, nctx):
    def M(bb_fn, use_ctx=True):
        return lambda: EventModel(bb_fn(), nctx, drop=DROPOUT, use_ctx=use_ctx)

    main = {
        "Plain Transformer": M(lambda: TransformerBB(nf)),
        "iTransformer (adapted)": M(lambda: iTransformerBB(nf)),
        "ModernTCN (adapted)": M(lambda: ModernTCNBB(nf)),
        "TimeMixer++-style (adapted)": M(lambda: TimeMixerPPBB(nf)),
        "Non-stationary Transformer (adapted)": M(lambda: NonstationaryBB(nf)),
        PROPOSED: M(lambda: TransformerBB(nf, multiscale=True, wavelet=True, attn_pool=True)),
    }
    ablation = {
        "A0 Plain Transformer": M(lambda: TransformerBB(nf)),
        "A1 + Multi-scale Temporal": M(lambda: TransformerBB(nf, multiscale=True)),
        "A2 + Multi-scale + Haar": M(lambda: TransformerBB(nf, multiscale=True,
                                                           wavelet=True)),
        "A3 + Attentive Pooling": M(lambda: TransformerBB(nf, multiscale=True,
                                                          wavelet=True, attn_pool=True)),
        "A4 - context branch (10 h only)": M(
            lambda: TransformerBB(nf, multiscale=True, wavelet=True, attn_pool=True),
            use_ctx=False),
    }
    return main, ablation


MAIN_MODELS, ABLATION_MODELS = make_registry(NKEEP, NCTX)

print("MODEL REGISTRY")
print("  every main-comparison model receives the context branch, so the proposed model")
print("  has no structural advantage; use_ctx=False appears only as ablation rung A4.\n")
for grp, reg in [("MAIN", MAIN_MODELS), ("ABLATION", ABLATION_MODELS)]:
    for k, fn in reg.items():
        print(f"  {grp:9s} {k:40s} params={sum(p.numel() for p in fn().parameters()):,}")

---
## Training

**Runtime warning.** As written the next cell trains 6 main models + 5 ablation rungs +
1 frozen-split model × 5 folds × 3 seeds ≈ **168 trainings**, roughly 8–19 h on a T4.
That will not survive one Colab session. Three reductions, in order:

1. `A0` and `A3` are architecturally identical to `Plain Transformer` and the proposed
   model. `REUSE_A0_A3 = True` (default) skips retraining them — saves 30 runs.
2. `ABLATION_SEEDS = [42]` (default) — saves 20 more.
3. Set `RESUME_FROM_CHECKPOINTS = True` in CELL 1 for every run *after* the first. Only
   while CELL 1 is unchanged; stale checkpoints load silently and will quietly invalidate
   the benchmark.

Time one fold first:

```python
import time
t0 = time.time()
run_grouped_cv(PROPOSED, MAIN_MODELS[PROPOSED], FOLDS[:1], seeds=[42])
print((time.time() - t0) / 60, "min")
```

Multiply by ~108 and decide before committing.


In [ ]:
# CELL 13 — Asset-grouped nested CV (primary) + frozen split (secondary)

def _avg_windows(frames):
    return (pd.concat(frames, ignore_index=True)
              .groupby(["event_id", "timestamp_ns"], as_index=False)
              .agg(asset_id=("asset_id", "first"), label=("label", "first"),
                   event_end_ns=("event_end_ns", "first"), logit=("logit", "mean")))


def run_grouped_cv(name, model_fn, folds, seeds=SEEDS, verbose=True, keep_windows=False):
    """
    PRIMARY evaluation. Per outer fold: test = fold assets, inner validation = next
    fold's assets, train = the rest. The operating threshold comes from each fold's own
    inner validation and is applied to that fold's test events; binary predictions are
    then pooled. Result: 58 out-of-fold events, 27 positives.
    """
    oof, hists, wt_all, wv_all = [], [], [], []

    for fi in range(len(folds)):
        te_a, va_a = folds[fi], folds[(fi + 1) % len(folds)]
        te = [i for i in range(len(STORE)) if ASSET_OF[i] in te_a]
        va = [i for i in range(len(STORE)) if ASSET_OF[i] in va_a]
        tr = [i for i in range(len(STORE)) if ASSET_OF[i] not in (te_a | va_a)]
        if verbose:
            print(f"  fold {fi}: train={len(tr)} val={len(va)} test={len(te)} events")

        vs, ts, wt, wv = [], [], [], []
        for sd in seeds:
            ck = CKPT_DIR / f"{safe_name(name)}__fold{fi}__seed{sd}.pt"
            m, h = train_bag_model(model_fn, tr, va, sd, ckpt=ck, log=verbose)
            h["fold"], h["seed"], h["model"] = fi, sd, name
            hists.append(h)

            if keep_windows:
                vev, vwin = score_events(m, va, return_windows=True)
                tev, twin = score_events(m, te, return_windows=True)
                wv.append(vwin); wt.append(twin)
            else:
                vev = score_events(m, va)
                tev = score_events(m, te)

            vs.append(vev.sort_values("event_id").reset_index(drop=True))
            ts.append(tev.sort_values("event_id").reset_index(drop=True))

            del m
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        ve = vs[0].copy(); ve["score"] = np.mean([d.score.to_numpy() for d in vs], axis=0)
        tf = ts[0].copy(); tf["score"] = np.mean([d.score.to_numpy() for d in ts], axis=0)

        tau = choose_threshold(ve.label.to_numpy(), ve.score.to_numpy())
        tf["pred"] = (tf.score.to_numpy() >= tau).astype(int)
        tf["fold"] = fi
        tf["threshold"] = tau
        oof.append(tf)

        if keep_windows:
            a = _avg_windows(wt); a["fold"] = fi; wt_all.append(a)
            b = _avg_windows(wv); b["fold"] = fi; wv_all.append(b)

        if verbose:
            print(f"    fold {fi}: tau={tau:+.4f}  "
                  f"test TP={int(((tf.label==1)&(tf.pred==1)).sum())} "
                  f"FN={int(((tf.label==1)&(tf.pred==0)).sum())} "
                  f"FP={int(((tf.label==0)&(tf.pred==1)).sum())} "
                  f"TN={int(((tf.label==0)&(tf.pred==0)).sum())}")

    oof = pd.concat(oof, ignore_index=True)
    oof["model"] = name
    hist = pd.concat(hists, ignore_index=True)

    if keep_windows:
        return oof, hist, pd.concat(wt_all, ignore_index=True), \
               pd.concat(wv_all, ignore_index=True)
    return oof, hist


def run_frozen_split(name, model_fn, seeds=SEEDS, verbose=True):
    """SECONDARY evaluation on the original frozen 32/13/13 asset-disjoint split."""
    tr = [i for i in range(len(STORE)) if STORE[i]["split"] == "train"]
    va = [i for i in range(len(STORE)) if STORE[i]["split"] == "val"]
    te = [i for i in range(len(STORE)) if STORE[i]["split"] == "test"]

    vs, ts = [], []
    for sd in seeds:
        ck = CKPT_DIR / f"{safe_name(name)}__frozen__seed{sd}.pt"
        m, _ = train_bag_model(model_fn, tr, va, sd, ckpt=ck, log=verbose)
        vs.append(score_events(m, va).sort_values("event_id").reset_index(drop=True))
        ts.append(score_events(m, te).sort_values("event_id").reset_index(drop=True))
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    ve = vs[0].copy(); ve["score"] = np.mean([d.score.to_numpy() for d in vs], axis=0)
    tf = ts[0].copy(); tf["score"] = np.mean([d.score.to_numpy() for d in ts], axis=0)
    tau = choose_threshold(ve.label.to_numpy(), ve.score.to_numpy())
    tf["pred"] = (tf.score.to_numpy() >= tau).astype(int)
    return ve, tf, tau


# ---- PRIMARY: main comparison -------------------------------------------------------
oof_bank, hist_bank, win_bank = {}, {}, {}

for nm, fn in MAIN_MODELS.items():
    print("\n" + "=" * 110)
    print("MAIN  " + nm)
    print("=" * 110)
    if nm == PROPOSED:
        o, h, wt, wv = run_grouped_cv(nm, fn, FOLDS, keep_windows=True)
        win_bank[nm] = {"test": wt, "val": wv}
    else:
        o, h = run_grouped_cv(nm, fn, FOLDS)
    oof_bank[nm], hist_bank[nm] = o, h

# ---- PRIMARY: ablation ladder -------------------------------------------------------
# A0 == Plain Transformer and A3 == the proposed model. With REUSE_A0_A3 = True those two
# rungs reuse the main-comparison runs (30 fewer trainings). State this in the caption.
REUSED = {"A0 Plain Transformer": "Plain Transformer",
          "A3 + Attentive Pooling": PROPOSED} if REUSE_A0_A3 else {}

abl_bank, abl_hist = {}, {}
for nm, fn in ABLATION_MODELS.items():
    if nm in REUSED:
        src = REUSED[nm]
        abl_bank[nm] = oof_bank[src].copy()
        abl_bank[nm]["model"] = nm
        abl_hist[nm] = hist_bank[src].copy()
        abl_hist[nm]["model"] = nm
        print(f"\nABLATION  {nm}  <- reusing the '{src}' runs (identical architecture)")
        continue
    print("\n" + "=" * 110)
    print("ABLATION  " + nm)
    print("=" * 110)
    o, h = run_grouped_cv(nm, fn, FOLDS, seeds=ABLATION_SEEDS)
    abl_bank[nm], abl_hist[nm] = o, h

# ---- SECONDARY: frozen 13-event split, proposed model only --------------------------
print("\n" + "=" * 110)
print("SECONDARY  frozen 32/13/13 split  —  " + PROPOSED)
print("=" * 110)
frozen_val, frozen_test, frozen_tau = run_frozen_split(PROPOSED, MAIN_MODELS[PROPOSED])

pd.concat(oof_bank.values(), ignore_index=True).to_csv(
    RESULT / "OOF_event_scores_main.csv", index=False)
pd.concat(abl_bank.values(), ignore_index=True).to_csv(
    RESULT / "OOF_event_scores_ablation.csv", index=False)
pd.concat(list(hist_bank.values()) + list(abl_hist.values()), ignore_index=True).to_csv(
    RESULT / "training_histories.csv", index=False)
frozen_test.to_csv(RESULT / "FROZEN_split_test_event_scores.csv", index=False)
win_bank[PROPOSED]["test"].to_csv(RESULT / "OOF_window_logits_proposed_test.csv", index=False)

print("\nAll models trained. Event scores and window logits saved to", RESULT)

In [ ]:
# CELL 14 — Pooled metrics, cluster bootstrap, permutation tests

def pooled_metrics(oof):
    """
    PR-AUC / ROC-AUC pool scores produced by different folds. Report them as a pooled
    ranking summary of out-of-fold discrimination, not as a single calibrated model AUC.
    Threshold metrics are exact: each event was classified by its own fold's threshold.
    """
    y = oof.label.to_numpy(int)
    s = oof.score.to_numpy(float)
    p = oof.pred.to_numpy(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n_events": int(len(oof)), "n_pos": int(y.sum()),
        "PR-AUC": float(average_precision_score(y, s)),
        "ROC-AUC": float(roc_auc_score(y, s)),
        "Precision": float(precision_score(y, p, zero_division=0)),
        "Recall": float(recall_score(y, p, zero_division=0)),
        "F1": float(f1_score(y, p, zero_division=0)),
        "MCC": float(matthews_corrcoef(y, p)),
        "Balanced Accuracy": float(balanced_accuracy_score(y, p)),
        "Accuracy": float(accuracy_score(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def cluster_bootstrap(oof, n_boot=5000, seed=20260824):
    """Resample ASSETS: events from one turbine are not independent units."""
    rng = np.random.default_rng(seed)
    assets = oof.asset_id.unique()
    by = {a: oof[oof.asset_id == a] for a in assets}
    acc = {"PR-AUC": [], "ROC-AUC": [], "F1": [], "MCC": [], "Recall": []}
    for _ in range(n_boot):
        b = pd.concat([by[a] for a in rng.choice(assets, len(assets), replace=True)],
                      ignore_index=True)
        if b.label.nunique() < 2:
            continue
        acc["PR-AUC"].append(average_precision_score(b.label, b.score))
        acc["ROC-AUC"].append(roc_auc_score(b.label, b.score))
        acc["F1"].append(f1_score(b.label, b.pred, zero_division=0))
        acc["MCC"].append(matthews_corrcoef(b.label, b.pred))
        acc["Recall"].append(recall_score(b.label, b.pred, zero_division=0))
    out = {"n_valid_boot": len(acc["PR-AUC"])}
    for k, v in acc.items():
        out[f"{k} CI low"] = float(np.quantile(v, 0.025))
        out[f"{k} CI high"] = float(np.quantile(v, 0.975))
    return out


def paired_permutation(oof_a, oof_b, metric="PR-AUC", n_perm=10000, seed=1):
    """Asset-level paired permutation test for 'A discriminates better than B'."""
    a = oof_a.sort_values("event_id").reset_index(drop=True)
    b = oof_b.sort_values("event_id").reset_index(drop=True)
    assert (a.event_id.to_numpy() == b.event_id.to_numpy()).all()
    fn = average_precision_score if metric == "PR-AUC" else roc_auc_score
    y = a.label.to_numpy(int)
    sa, sb = a.score.to_numpy(float), b.score.to_numpy(float)
    obs = fn(y, sa) - fn(y, sb)
    rng = np.random.default_rng(seed)
    assets = a.asset_id.unique()
    cnt = 0
    for _ in range(n_perm):
        flip = a.asset_id.map({t: bool(rng.integers(0, 2)) for t in assets}).to_numpy()
        pa = np.where(flip, sb, sa)
        pb = np.where(flip, sa, sb)
        if abs(fn(y, pa) - fn(y, pb)) >= abs(obs):
            cnt += 1
    return obs, (cnt + 1) / (n_perm + 1)


results = pd.DataFrame([{"Model": nm, **pooled_metrics(o), **cluster_bootstrap(o)}
                        for nm, o in oof_bank.items()])
results = results.sort_values("PR-AUC", ascending=False).reset_index(drop=True)

abl_results = pd.DataFrame([{"Model": nm, **pooled_metrics(o), **cluster_bootstrap(o)}
                            for nm, o in abl_bank.items()])
abl_results["order"] = [list(ABLATION_MODELS).index(m) for m in abl_results.Model]
abl_results = abl_results.sort_values("order").drop(columns="order").reset_index(drop=True)

# Per-seed stability from the INNER-VALIDATION criterion, so no test peeking.
# Resumed runs return empty histories, hence the guard.
_h = [d for d in hist_bank.values() if len(d) and "criterion" in d.columns]
if _h:
    stability = (pd.concat(_h, ignore_index=True)
                   .sort_values(["model", "fold", "seed", "epoch"])
                   .groupby(["model", "fold", "seed"], as_index=False)
                   .agg(best_criterion=("criterion", "max"),
                        best_val_event_roc=("val_event_roc", "max"),
                        epochs_run=("epoch", "max"))
                   .groupby("model", as_index=False)
                   .agg(crit_mean=("best_criterion", "mean"),
                        crit_sd=("best_criterion", "std"),
                        val_ev_roc_mean=("best_val_event_roc", "mean"),
                        val_ev_roc_sd=("best_val_event_roc", "std"),
                        mean_epochs=("epochs_run", "mean")))
else:
    stability = pd.DataFrame(columns=["model", "crit_mean", "crit_sd",
                                      "val_ev_roc_mean", "val_ev_roc_sd", "mean_epochs"])
    print("NOTE: no training histories available (resumed run) - stability table skipped.")

frozen_metrics = metrics_from_event_df(frozen_test, tau=frozen_tau)

results.to_csv(RESULT / "MAIN_pooled_metrics_asset_bootstrap_CI.csv", index=False)
abl_results.to_csv(RESULT / "ABLATION_pooled_metrics.csv", index=False)
stability.to_csv(RESULT / "seed_stability_inner_validation.csv", index=False)

print("MAIN — 58 out-of-fold events, asset-grouped, 27 positives")
print(results[["Model", "PR-AUC", "PR-AUC CI low", "PR-AUC CI high",
               "ROC-AUC", "Recall", "Precision", "F1", "MCC",
               "TP", "FP", "TN", "FN"]].to_string(index=False))

print("\nPAIRED PERMUTATION TESTS vs " + PROPOSED)
for nm in oof_bank:
    if nm != PROPOSED:
        d, p = paired_permutation(oof_bank[PROPOSED], oof_bank[nm])
        print(f"  vs {nm:40s} dPR-AUC={d:+.3f}  p={p:.4f}")

print("\nSECONDARY — frozen 13-event split, " + PROPOSED)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in frozen_metrics.items()})

In [ ]:
# CELL 15 — Figure helpers, FIG1 main PR, FIG8 duration control

PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#6A3D9A", "#8B4513"]
DASHES = ["--", "-.", ":", (0, (5, 2)), (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (6, 2, 1, 2))]


def style_for(names, highlight):
    out, k = {}, 0
    for n in names:
        if n == highlight:
            out[n] = ("#D55E00", "-", 3.3, 10)
        else:
            out[n] = (PALETTE[k % len(PALETTE)], DASHES[k % len(DASHES)], 1.9, 3)
            k += 1
    return out


def journal_axes(ax, grid=True):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(direction="in", length=5, width=0.9)
    if grid:
        ax.grid(True, linestyle="--", linewidth=0.55, alpha=0.18)
    ax.set_axisbelow(True)


def save_figure(fig, stem, tiff=False):
    fig.savefig(RESULT / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(RESULT / f"{stem}.png", dpi=600, bbox_inches="tight")
    if tiff:
        fig.savefig(RESULT / f"{stem}.tiff", dpi=600, bbox_inches="tight")
    print("saved:", stem)


def fig_curves(bank, names, kind, stem, highlight=PROPOSED, figsize=(7.3, 5.6)):
    """
    Step curves, not line plots: with a small discrete event set, interpolating between
    operating points draws points that do not exist.
    """
    st = style_for(names, highlight)
    fig, ax = plt.subplots(figsize=figsize)
    for nm in names:
        d = bank[nm]
        col, ls, lw, z = st[nm]
        y, s = d.label.to_numpy(int), d.score.to_numpy(float)
        if kind == "pr":
            p, r, _ = precision_recall_curve(y, s)
            ax.step(r, p, where="post", color=col, linestyle=ls, linewidth=lw, zorder=z,
                    label=f"{nm} (AP={average_precision_score(y, s):.3f})")
        else:
            fpr, tpr, _ = roc_curve(y, s)
            ax.step(fpr, tpr, where="post", color=col, linestyle=ls, linewidth=lw, zorder=z,
                    label=f"{nm} (AUC={roc_auc_score(y, s):.3f})")

    if kind == "pr":
        prev = bank[names[0]].label.mean()
        ax.axhline(prev, color="#777777", linestyle=(0, (4, 3)), linewidth=1.2,
                   label=f"Event prevalence ({prev:.3f})")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.legend(loc="lower left", frameon=False, handlelength=3.2, labelspacing=0.5)
    else:
        ax.plot([0, 1], [0, 1], color="#777777", linestyle=(0, (4, 3)), linewidth=1.2,
                label="Chance")
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.legend(loc="lower right", frameon=False, handlelength=3.2, labelspacing=0.5)

    ax.set_xlim(0, 1.01); ax.set_ylim(0, 1.03)
    journal_axes(ax)
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()


def fig_duration_control(oof, stem="FIG8_duration_control"):
    """
    Anomaly events average 375.7 h vs 318.8 h for normal events, so a reviewer will ask
    whether duration alone explains the result. Answer it in the paper.
    """
    dmap = {s["event_id"]: (s["event_end_ns"] - s["event_start_ns"]) / 3.6e12 for s in STORE}
    d = oof.assign(duration_h=oof.event_id.map(dmap))
    auc_dur = roc_auc_score(d.label, d.duration_h)
    auc_mod = roc_auc_score(d.label, d.score)

    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    for lab, mk, col in [(0, "o", "#222222"), (1, "s", "#D55E00")]:
        m = d.label == lab
        ax.scatter(d.duration_h[m], d.score[m], marker=mk, s=62, facecolor="white",
                   edgecolor=col, linewidth=1.4, label="Normal" if lab == 0 else "Fault")
    ax.set_xlabel("Event duration (h)")
    ax.set_ylabel("Out-of-fold event risk score")
    ax.set_title(f"duration-only AUC = {auc_dur:.3f}     model AUC = {auc_mod:.3f}")
    ax.legend(frameon=False)
    journal_axes(ax)
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()
    return auc_dur, auc_mod


MAIN_NAMES = list(MAIN_MODELS)
fig_curves(oof_bank, MAIN_NAMES, "pr", "FIG1_event_PR_main")
auc_dur, auc_mod = fig_duration_control(oof_bank[PROPOSED])

In [ ]:
# CELL 16 — FIG2 main ROC

fig_curves(oof_bank, MAIN_NAMES, "roc", "FIG2_event_ROC_main")

In [ ]:
# CELL 17 — Ablation table, FIG3 PR, FIG4 ROC

ABL_NAMES = list(ABLATION_MODELS)

ablation_table = abl_results[[
    "Model", "n_events", "n_pos",
    "PR-AUC", "PR-AUC CI low", "PR-AUC CI high",
    "ROC-AUC", "ROC-AUC CI low", "ROC-AUC CI high",
    "Precision", "Recall", "F1", "MCC",
    "TP", "FP", "TN", "FN",
]].copy()
ablation_table.to_csv(RESULT / "ABLATION_table.csv", index=False)

print("ABLATION LADDER (58 out-of-fold events)")
print(ablation_table.to_string(index=False))
print("\nReport this ladder as measured. Non-monotonic rungs are a result, not a defect:")
print("in the earlier V3 run the wavelet variant beat the full model on PR-AUC while the")
print("full model won ROC-AUC, recall, F1 and MCC.")

fig_curves(abl_bank, ABL_NAMES, "pr", "FIG3_ablation_PR",
           highlight="A3 + Attentive Pooling", figsize=(7.0, 5.4))
fig_curves(abl_bank, ABL_NAMES, "roc", "FIG4_ablation_ROC",
           highlight="A3 + Attentive Pooling", figsize=(7.0, 5.4))

In [ ]:
# CELL 18 — FIG5 metric comparison (dot plot)

# Horizontal dot plot: a 19x10 inch grouped bar chart at font size 20 reads as a slide.
order = list(reversed(MAIN_NAMES))
md = results.set_index("Model").loc[order].reset_index()

fig, ax = plt.subplots(figsize=(7.6, 5.2))
ypos = np.arange(len(md))

for col, lab, mk in [("Recall", "Recall", "o"), ("F1", "F1", "s"), ("MCC", "MCC", "D")]:
    ax.scatter(md[col], ypos, marker=mk, s=76, facecolor="white",
               edgecolor="#222222", linewidth=1.35, label=lab, zorder=5)

hl = np.flatnonzero(md.Model.to_numpy() == PROPOSED)
if len(hl):
    ax.axhspan(hl[0] - 0.42, hl[0] + 0.42, color="#D55E00", alpha=0.08, zorder=0)

ax.set_yticks(ypos)
ax.set_yticklabels(md.Model)
ax.set_xlabel("Score (58 out-of-fold events)")
ax.set_xlim(0, 1.02)
journal_axes(ax)
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, ncol=3, loc="lower right")
fig.tight_layout()
save_figure(fig, "FIG5_metric_comparison")
plt.show()

In [ ]:
# CELL 19 — FIG6 confusion matrices

def plot_confusion(oof, title, stem):
    """
    Built from oof.pred, which already carries each fold's own inner-validation
    threshold. There is no single global tau to re-apply, and no number is typed by hand.
    """
    cm = confusion_matrix(oof.label.to_numpy(int), oof.pred.to_numpy(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5.6, 5.1))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=max(1, cm.max()))
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=24, fontweight="bold",
                    color="white" if cm[i, j] > 0.55 * cm.max() else "#111111")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "Fault event"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Fault event"])
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    ax.set_title(f"{title}\n{len(oof)} events "
                 f"({int((oof.label == 0).sum())} normal, {int(oof.label.sum())} fault)")
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()
    return cm


cm_prop = plot_confusion(oof_bank[PROPOSED],
                         f"{PROPOSED} — pooled out-of-fold",
                         "FIG6A_proposed_confusion")

best_model = str(results.iloc[0]["Model"])
cm_best = plot_confusion(oof_bank[best_model],
                         f"Best pooled PR-AUC: {best_model}",
                         "FIG6B_best_model_confusion")

cm_frozen = plot_confusion(frozen_test,
                           f"{PROPOSED} — frozen 13-event split (secondary)",
                           "FIG6C_frozen_split_confusion")

print("proposed, pooled 58 events:\n", cm_prop)
print("\nfrozen 13-event split:\n", cm_frozen)

In [ ]:
# CELL 20 — Operational rolling-6h alarm and lead-time evaluation

# Verify the lead-time reference point BEFORE trusting any number below.
_chk = event_info.assign(dur_h=(event_info.event_end_dt - event_info.event_start_dt)
                         .dt.total_seconds() / 3600.0)
print("EVENT FRAME DURATION BY CLASS (h)")
print(_chk.groupby("is_anomaly").dur_h.describe()[["count", "mean", "50%", "min", "max"]]
      .to_string())
print("\nlead_hours is measured as (event_end - first_alarm). If event_end is only the end")
print("of the CSV prediction section rather than the recorded fault time, lead times are")
print("inflated by the post-fault tail and must be restated in the manuscript.\n")


def rolling_risk(win_df, window=OP_WINDOW, q=TOP_FRACTION):
    """Rolling 6-h tail-mean of window logits, per event."""
    out = {}
    for eid, g in win_df.groupby("event_id", sort=True):
        g = g.sort_values("timestamp_ns").reset_index(drop=True)
        v = g.logit.to_numpy()
        risk = np.empty(len(v))
        for i in range(len(v)):
            seg = np.sort(v[max(0, i - window + 1):i + 1])
            k = max(1, int(math.ceil(q * len(seg))))
            risk[i] = seg[-k:].mean()
        out[int(eid)] = g.assign(rolling_risk=risk)
    return out


def _event_stat(roll, tail_q=OP_TAIL_Q):
    """Tail quantile, not max: a maximum over a longer sequence is larger for free."""
    return pd.DataFrame([{"event_id": e,
                          "asset_id": g.asset_id.iloc[0],
                          "label": int(g.label.iloc[0]),
                          "score": float(np.quantile(g.rolling_risk, tail_q))}
                         for e, g in roll.items()])


def operational_report(win_test, win_val):
    """
    Nested: within each fold the alarm threshold is derived from that fold's inner
    validation events, then applied to that fold's test events. Results are pooled.
    """
    ev_rows, lead_rows = [], []

    for fi in sorted(win_test.fold.unique()):
        rt = rolling_risk(win_test[win_test.fold == fi])
        rv = rolling_risk(win_val[win_val.fold == fi])

        vstat = _event_stat(rv)
        tau = choose_threshold(vstat.label.to_numpy(), vstat.score.to_numpy())

        tstat = _event_stat(rt)
        tstat["pred"] = (tstat.score >= tau).astype(int)
        tstat["fold"] = fi
        tstat["threshold"] = tau
        ev_rows.append(tstat)

        for e, g in rt.items():
            cr = g[g.rolling_risk >= tau]
            lead, first = np.nan, pd.NaT
            if len(cr):
                fns = int(cr.iloc[0].timestamp_ns)
                first = pd.to_datetime(fns)
                lead = (int(g.event_end_ns.iloc[0]) - fns) / 3.6e12
            lead_rows.append({
                "event_id": e, "asset_id": g.asset_id.iloc[0],
                "label": int(g.label.iloc[0]), "fold": fi,
                "alarmed": bool(len(cr) > 0), "first_alarm": first,
                "lead_hours": lead,
                "observed_hours": (g.timestamp_ns.max() - g.timestamp_ns.min()) / 3.6e12,
                "threshold": tau,
            })

    op_ev = pd.concat(ev_rows, ignore_index=True)
    lead_df = pd.DataFrame(lead_rows)

    an = lead_df[lead_df.label == 1]
    no = lead_df[lead_df.label == 0]
    horizons = np.array([1, 3, 6, 12, 24, 48, 72], dtype=float)
    frac = np.array([float(((an.alarmed) & (an.lead_hours >= h)).mean()) for h in horizons])

    # exposure-normalised false alarms, not "n of N events"
    fa_100d = (no.alarmed.sum() / max(no.observed_hours.sum() / 24.0, 1e-9)) * 100.0

    summary = {
        "faults_detected": f"{int(an.alarmed.sum())}/{len(an)}",
        "normal_events_with_alarm": f"{int(no.alarmed.sum())}/{len(no)}",
        "false_alarms_per_100_normal_event_days": round(float(fa_100d), 3),
        "median_lead_h": (round(float(an.loc[an.alarmed, "lead_hours"].median()), 2)
                          if an.alarmed.any() else None),
        "min_lead_h": (round(float(an.loc[an.alarmed, "lead_hours"].min()), 2)
                       if an.alarmed.any() else None),
        "max_lead_h": (round(float(an.loc[an.alarmed, "lead_hours"].max()), 2)
                       if an.alarmed.any() else None),
        **{f"detected_ge_{int(h)}h": round(float(f), 4) for h, f in zip(horizons, frac)},
    }
    return lead_df, op_ev, summary, horizons, frac


lead_df, op_ev, op_sum, horizons, fractions = operational_report(
    win_bank[PROPOSED]["test"], win_bank[PROPOSED]["val"])

op_metrics = metrics_from_event_df(op_ev, pred_col="pred")

lead_df.to_csv(RESULT / "OP_lead_times.csv", index=False)
pd.DataFrame({"lead_h": horizons, "detected_fraction": fractions}).to_csv(
    RESULT / "OP_leadtime_curve.csv", index=False)
with open(RESULT / "OP_summary.json", "w") as f:
    json.dump(op_sum, f, indent=2, default=str)

print("OPERATIONAL ROLLING-6H ANALYSIS — " + PROPOSED)
print(json.dumps(op_sum, indent=2, default=str))
print("\nevent-level metrics at the operational threshold:")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in op_metrics.items()})

fig, ax = plt.subplots(figsize=(7.2, 5.1))
x = np.arange(len(horizons))
ax.plot(x, 100 * fractions, color="#D55E00", linewidth=2.8, marker="o", markersize=7,
        markerfacecolor="white", markeredgecolor="#D55E00", markeredgewidth=1.8)
for xi, v in zip(x, 100 * fractions):
    ax.annotate(f"{v:.1f}%", (xi, v), xytext=(0, 9), textcoords="offset points",
                ha="center", fontsize=10)
ax.axvline(list(horizons).index(6.0), color="#555555", linestyle=(0, (4, 3)), linewidth=1.1)
ax.set_xticks(x)
ax.set_xticklabels([str(int(h)) for h in horizons])
ax.set_xlabel("Minimum warning lead time (h)")
ax.set_ylabel("Detected fault events (%)")
ax.set_ylim(0, 103)
journal_axes(ax)
fig.tight_layout()
save_figure(fig, "FIG7_lead_time_curve")
plt.show()

In [ ]:
# CELL 21 — Save event scores and protocol metadata

for nm, o in {**oof_bank, **abl_bank}.items():
    o.to_csv(RESULT / f"OOF_event_scores_{safe_name(nm)}.csv", index=False)

frozen_val.to_csv(RESULT / "FROZEN_val_event_scores_proposed.csv", index=False)

protocol = {
    "protocol_version": PROTOCOL_VERSION,
    "dataset": "CARE to Compare v6",
    "wind_farm": "C",
    "n_events": int(len(event_info)),
    "n_anomaly": int(event_info.is_anomaly.sum()),
    "n_normal": int((~event_info.is_anomaly).sum()),
    "n_assets": int(event_info.asset_id.nunique()),
    "input_channels_available": int(NF),
    "input_channels_used": int(NKEEP),
    "channel_hygiene_rules": ["degenerate baseline scale",
                              "cumulative/counter channels",
                              "clipping-rate label leakage"],
    "input": f"{LOOKBACK} x {NKEEP} 10-min window (10 h) + {NCTX}-dim baseline-deviation "
             f"context at 1 d / 7 d / 30 d",
    "normalisation": f"per-event historical healthy baseline, robust median/IQR, "
                     f"clipped at +/-{CLIP}",
    "status_policy": "status_type_id is never a model feature; abnormal-status endpoints "
                     "are excluded from the event frame",
    "training_objective": f"event-bag MIL: BCE(mean of top-{int(TOP_FRACTION*100)}% window "
                          f"logits, y_event) + {LAMBDA_NEG} * BCE(window logits, 0) on "
                          f"normal events only",
    "event_aggregation": f"mean of top {int(TOP_FRACTION*100)}% window logits "
                         f"(identical to the training statistic)",
    "checkpoint_selection": "0.7 * val event ROC-AUC + 0.3 * val window ROC-AUC, EMA weights",
    "operating_threshold": "inner validation only, 0.7 * MCC + 0.3 * F1",
    "primary_evaluation": f"{N_FOLDS}-fold asset-grouped nested CV over all 58 events; "
                          f"threshold from each fold's own inner validation",
    "secondary_evaluation": "frozen 32/13/13 asset-disjoint split",
    "uncertainty": "cluster bootstrap over assets (5000) + asset-level paired "
                   "permutation test (10000)",
    "operational_analysis": f"rolling {OP_WINDOW}-sample (6 h) tail-mean risk; event "
                            f"statistic = quantile {OP_TAIL_Q}; false alarms reported per "
                            f"100 normal-event days",
    "duration_control": {"duration_only_ROC_AUC": round(float(auc_dur), 4),
                         "model_ROC_AUC": round(float(auc_mod), 4)},
    "seeds": SEEDS,
    "test_role": "reporting only; never used for checkpoint, threshold or hyperparameter "
                 "selection",
    "modern_baseline_note": "Named modern architectures are task adaptations evaluated "
                            "under one common CARE protocol, not byte-identical "
                            "reproductions of external repositories.",
    "sample_size_note": "The independent test units are events (58 out-of-fold, 27 "
                        "positive). Windows are repeated overlapping observations of "
                        "those events and must never be reported as a sample size.",
}

with open(RESULT / "protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

print("Saved all event score files and protocol.json to", RESULT)

In [ ]:
# CELL 22 — Final report

print("=" * 135)
print("CARE WIND FARM C — EVENT-LEVEL EARLY FAULT DETECTION")
print("=" * 135)

print("\nDATA")
print("  events              :", len(event_info))
print("  anomaly / normal    :", int(event_info.is_anomaly.sum()), "/",
      int((~event_info.is_anomaly).sum()))
print("  assets              :", event_info.asset_id.nunique())
print("  channels available  :", NF)
print("  channels used       :", NKEEP, "(after hygiene)")
print("  window              :", f"{LOOKBACK} x {NKEEP} = 10.0 h")
print("  context             :", f"{NCTX} dims (1 d / 7 d / 30 d baseline deviation)")
print("  seeds per fold      :", len(SEEDS))

print("\nPRIMARY — 5-fold asset-grouped nested CV, 58 events, 27 positive")
cols = ["Model", "PR-AUC", "PR-AUC CI low", "PR-AUC CI high",
        "ROC-AUC", "ROC-AUC CI low", "ROC-AUC CI high",
        "Precision", "Recall", "F1", "MCC",
        "Balanced Accuracy", "Accuracy", "TP", "FP", "TN", "FN"]
print(results[cols].to_string(index=False))

print("\nABLATION")
print(ablation_table.to_string(index=False))

print("\nSEED / FOLD STABILITY (inner-validation criterion, no test peeking)")
if len(stability):
    print(stability.sort_values("crit_mean", ascending=False).to_string(index=False))
else:
    print("  unavailable on a resumed run")

print("\nDURATION CONTROL")
print(f"  duration-only ROC-AUC : {auc_dur:.4f}")
print(f"  model ROC-AUC         : {auc_mod:.4f}")
if auc_dur > 0.65:
    print("  WARNING: event duration alone is informative. Report this control explicitly.")

ours = results[results.Model == PROPOSED].iloc[0]
rank = int((results["PR-AUC"] > ours["PR-AUC"]).sum() + 1)

print(f"\n{PROPOSED}")
for k in ["PR-AUC", "PR-AUC CI low", "PR-AUC CI high",
          "ROC-AUC", "ROC-AUC CI low", "ROC-AUC CI high",
          "Precision", "Recall", "F1", "MCC", "Balanced Accuracy", "Accuracy"]:
    print(f"  {k:22s}: {float(ours[k]):.4f}")
print(f"  {'pooled PR-AUC rank':22s}: {rank}/{len(results)}")

print("\nSECONDARY — frozen 13-event split")
for k, v in frozen_metrics.items():
    print(f"  {k:22s}: {v:.4f}" if isinstance(v, float) else f"  {k:22s}: {v}")

print("\nOPERATIONAL ROLLING-6H")
for k, v in op_sum.items():
    print(f"  {k:42s}: {v}")

if rank == 1:
    print(f"\nRANK STATEMENT: {PROPOSED} is #1 of {len(results)} by pooled PR-AUC.")
    print("Rank 1 is NOT significance. Quote the paired permutation p-values and note")
    print("that the 95% asset-bootstrap intervals overlap before making any claim.")
else:
    print(f"\nRANK STATEMENT: {PROPOSED} is #{rank} of {len(results)} by pooled PR-AUC.")
    print("Do NOT claim numerical SOTA from this benchmark.")

print("\nSAMPLE-SIZE STATEMENT FOR THE MANUSCRIPT")
print("  58 independent out-of-fold events (27 positive) from 22 asset-disjoint turbines.")
print("  Window counts are repeated, overlapping observations of those events and are")
print("  never reported as a sample size.")

print("\nOUTPUT DIRECTORY:", RESULT)
print("\nGENERATED FILES")
for p in sorted(RESULT.iterdir()):
    if p.is_file():
        print("  -", p.name)

print("=" * 135)
print("BENCHMARK COMPLETE")
print("=" * 135)

---
## CELL 23 — Rank ensemble + nested deep/classical blend (read-only, no retraining)

Your own diagnostics (D5/D6 in CELL 6D) showed that a plain classical baseline on
hand-crafted features reaches ROC-AUC 0.687 / PR-AUC 0.690, and that this signal is real
(D6 clears the permutation null, p=0.020). Every neural architecture in CELL 13's main
comparison currently lands at or below that classical number — including the proposed
model. Training curves in the same run show textbook overfitting (loss collapses in ~4
epochs, validation event ROC-AUC never tracks it), which is expected with 29-42 training
events at this model capacity.

This cell does not retrain anything. It reads the CSVs CELL 13/14/6D already saved and:

1. Selects ensemble members by their **inner-validation** criterion only
   (`seed_stability_inner_validation.csv`) — never by test/OOF performance.
2. Averages those models' z-scored out-of-fold scores into one ensemble score per event.
3. Blends the ensemble with the classical baseline using a weight chosen **per test fold
   from the other four folds only** — nested, so no fold's own labels ever influence its
   own blend weight or threshold. This is asserted programmatically, not just described:
   the cell raises if any test-fold index is found in that fold's own selection subset.
4. Reports the full comparison — every individual model, the classical baseline, the
   ensemble, and the blend — with asset-cluster bootstrap CIs and a paired permutation
   test against the best prior component.

**If the blend does not beat every component, the cell says so.** That is itself a
reportable finding: it would mean the deep architectures are not yet extracting more from
234 SCADA channels than robust per-channel quantile summaries do at n=58 events.


In [ ]:
# =====================================================================================
# CELL 23 — NEW: rank ensemble of deep models + nested deep/classical blend
#
# READS ONLY existing saved CSVs. Trains nothing, touches no checkpoint.
# Run this AFTER CELL 22 has completed at least once (needs its saved CSVs on Drive).
#
# What this does, in order:
#   1. Load every model's pooled out-of-fold event scores (CELL 13/14 outputs) and the
#      classical D5 baseline (CELL 6D output).
#   2. Select ensemble members by their INNER-VALIDATION criterion only
#      (seed_stability_inner_validation.csv) - never by test/OOF performance. Uses the
#      top half by crit_mean.
#   3. Z-score and average the selected models' out-of-fold scores per event.
#   4. Blend that ensemble with the classical baseline using a weight lambda chosen
#      PER TEST FOLD from the other four folds only (nested - no fold ever sees its own
#      labels when its own lambda/threshold is picked).
#   5. Report pooled metrics + asset-cluster bootstrap CI for: best single deep model,
#      the ensemble, the classical baseline alone, and the blend - so the comparison is
#      complete and nothing is cherry-picked out of the table.
#
# If the blend does not beat both components, that is reported as-is. The rank
# statement and paired-permutation tests from CELL 22 are recomputed and printed again
# so a partial improvement cannot be presented as a headline result if it is not one.
# =====================================================================================

MAIN_CSV = RESULT / "OOF_event_scores_main.csv"
D5_CSV = RESULT / "D5_classical_baseline_oof.csv"
STAB_CSV = RESULT / "seed_stability_inner_validation.csv"

for p in (MAIN_CSV, D5_CSV, STAB_CSV):
    assert p.exists(), f"Missing required file: {p}\nRun CELL 6D and CELL 13/14 first."

main_oof = pd.read_csv(MAIN_CSV)
d5 = pd.read_csv(D5_CSV)
stab = pd.read_csv(STAB_CSV)

print("=" * 100)
print("STEP 1 - inputs loaded")
print("=" * 100)
print(f"main_oof: {main_oof.model.nunique()} models x {main_oof.event_id.nunique()} events")
print(f"D5 columns: {[c for c in d5.columns if c.startswith('oof_')]}")


# ---- fold assignment: recovered from asset_folds.csv, same folds used everywhere ----
asset_fold_map = pd.read_csv(RESULT / "asset_folds.csv").set_index("asset_id")["fold"].to_dict()


def fold_of(asset_id):
    return asset_fold_map[asset_id]


def zscore(s):
    s = s.astype(float)
    sd = s.std()
    return (s - s.mean()) / (sd if sd > 1e-9 else 1.0)


def choose_threshold(y, s):
    y = np.asarray(y, int)
    s = np.asarray(s, float)
    u = np.unique(s)
    cand = (u[:-1] + u[1:]) / 2.0 if len(u) > 1 else u
    if len(cand) == 0:
        return float(s.mean())
    best = (-2.0, float(cand[0]))
    for t in cand:
        p = (s >= t).astype(int)
        v = 0.7 * matthews_corrcoef(y, p) + 0.3 * f1_score(y, p, zero_division=0)
        if v > best[0]:
            best = (v, float(t))
    return best[1]


def pooled_metrics_df(oof):
    y = oof.label.to_numpy(int)
    s = oof.score.to_numpy(float)
    p = oof.pred.to_numpy(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "PR-AUC": float(average_precision_score(y, s)),
        "ROC-AUC": float(roc_auc_score(y, s)),
        "Precision": float(precision_score(y, p, zero_division=0)),
        "Recall": float(recall_score(y, p, zero_division=0)),
        "F1": float(f1_score(y, p, zero_division=0)),
        "MCC": float(matthews_corrcoef(y, p)),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }


def cluster_bootstrap_ci(oof, n_boot=5000, seed=20260824):
    rng = np.random.default_rng(seed)
    assets = oof.asset_id.unique()
    by = {a: oof[oof.asset_id == a] for a in assets}
    pr, roc = [], []
    for _ in range(n_boot):
        b = pd.concat([by[a] for a in rng.choice(assets, len(assets), replace=True)],
                      ignore_index=True)
        if b.label.nunique() < 2:
            continue
        pr.append(average_precision_score(b.label, b.score))
        roc.append(roc_auc_score(b.label, b.score))
    return {"PR-AUC CI low": float(np.quantile(pr, .025)),
            "PR-AUC CI high": float(np.quantile(pr, .975)),
            "ROC-AUC CI low": float(np.quantile(roc, .025)),
            "ROC-AUC CI high": float(np.quantile(roc, .975)),
            "n_valid_boot": len(pr)}


# =====================================================================================
# STEP 2 - select ensemble members by INNER-VALIDATION criterion only
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 2 - ensemble member selection (inner-validation criterion, no test peeking)")
print("=" * 100)

if len(stab) == 0:
    raise RuntimeError(
        "seed_stability_inner_validation.csv is empty - this happens on a fully resumed "
        "run where train_bag_model returned no history. Rerun CELL 13 with "
        "RESUME_FROM_CHECKPOINTS = False at least once to regenerate it."
    )

median_crit = stab.crit_mean.median()
ensemble_members = stab.loc[stab.crit_mean >= median_crit, "model"].tolist()
ensemble_members = [m for m in ensemble_members if m in main_oof.model.unique()]

print(stab.sort_values("crit_mean", ascending=False)[
    ["model", "crit_mean", "crit_sd", "val_ev_roc_mean"]].to_string(index=False))
print(f"\nmedian crit_mean = {median_crit:.4f}")
print(f"selected for ensemble (>= median): {ensemble_members}")
assert len(ensemble_members) >= 2, "need at least 2 models to ensemble"


# =====================================================================================
# STEP 3 - z-score average -> deep ensemble OOF (one score per event)
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 3 - deep ensemble (z-scored average of selected models' OOF scores)")
print("=" * 100)

pivot = main_oof[main_oof.model.isin(ensemble_members)].pivot_table(
    index=["event_id", "asset_id", "label"], columns="model", values="score"
).reset_index()

for m in ensemble_members:
    pivot[f"z_{m}"] = zscore(pivot[m])

pivot["ensemble_z"] = pivot[[f"z_{m}" for m in ensemble_members]].mean(axis=1)
pivot["fold"] = pivot["asset_id"].map(fold_of)

ens_rows = []
for i in sorted(pivot.fold.unique()):
    tr = pivot.fold != i
    te = pivot.fold == i
    tau_i = choose_threshold(pivot.loc[tr, "label"].to_numpy(),
                             pivot.loc[tr, "ensemble_z"].to_numpy())
    sub = pivot.loc[te, ["event_id", "asset_id", "label"]].copy()
    sub["score"] = pivot.loc[te, "ensemble_z"].to_numpy()
    sub["pred"] = (sub["score"] >= tau_i).astype(int)
    sub["fold"] = i
    sub["threshold"] = tau_i
    ens_rows.append(sub)
ensemble_oof = pd.concat(ens_rows, ignore_index=True)

assert len(ensemble_oof) == main_oof.event_id.nunique()
assert set(ensemble_oof.event_id) == set(main_oof.event_id.unique())
print(f"ensemble OOF: {len(ensemble_oof)} events, covers every event exactly once (checked)")


# =====================================================================================
# STEP 4 - classical baseline OOF (best of the D5 models by its own OOF ROC-AUC)
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 4 - classical baseline reference")
print("=" * 100)

classical_cols = [c for c in d5.columns if c.startswith("oof_")]
best_classical_col = max(
    classical_cols, key=lambda c: roc_auc_score(d5.label, d5[c]))
print(f"best classical column: {best_classical_col} "
      f"(ROC-AUC {roc_auc_score(d5.label, d5[best_classical_col]):.3f})")

classical_oof = d5[["event_id", "asset_id", "label", best_classical_col]].rename(
    columns={best_classical_col: "score"}).copy()
classical_oof["fold"] = classical_oof["asset_id"].map(fold_of)
classical_rows = []
for i in sorted(classical_oof.fold.unique()):
    tr = classical_oof.fold != i
    te = classical_oof.fold == i
    tau_i = choose_threshold(classical_oof.loc[tr, "label"].to_numpy(),
                             classical_oof.loc[tr, "score"].to_numpy())
    sub = classical_oof.loc[te].copy()
    sub["pred"] = (sub["score"].to_numpy() >= tau_i).astype(int)
    sub["threshold"] = tau_i
    classical_rows.append(sub)
classical_oof_scored = pd.concat(classical_rows, ignore_index=True)


# =====================================================================================
# STEP 5 - NESTED deep-ensemble + classical blend
#
# For test fold i: search lambda on the OTHER FOUR folds only, then apply the winning
# lambda (and a threshold fit on those same four folds) to fold i. Fold i's own labels
# never participate in choosing its own lambda or its own threshold.
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 5 - nested deep-ensemble + classical blend (leak-checked)")
print("=" * 100)

blend_meta = ensemble_oof[["event_id", "asset_id", "label", "fold"]].merge(
    classical_oof_scored[["event_id", "score"]], on="event_id", how="inner"
).rename(columns={"score": "classical_raw"})

blend_meta["deep_z"] = zscore(
    ensemble_oof.set_index("event_id").loc[blend_meta.event_id, "score"].to_numpy())
blend_meta["classical_z"] = zscore(blend_meta["classical_raw"])

assert len(blend_meta) == len(ensemble_oof), "event mismatch between ensemble and classical OOF"

LAM_GRID = np.linspace(0.0, 1.0, 21)
_leak_check = []
blend_rows = []
for i in sorted(blend_meta.fold.unique()):
    tr_idx = blend_meta.index[blend_meta.fold != i]
    te_idx = blend_meta.index[blend_meta.fold == i]
    _leak_check.append(len(set(tr_idx) & set(te_idx)))  # must be 0 every time

    tr = blend_meta.loc[tr_idx]
    best = (-2.0, 0.5)
    for lam in LAM_GRID:
        s_tr = lam * tr["deep_z"] + (1 - lam) * tr["classical_z"]
        tau = choose_threshold(tr["label"].to_numpy(), s_tr.to_numpy())
        pred = (s_tr.to_numpy() >= tau).astype(int)
        val = (0.7 * matthews_corrcoef(tr["label"], pred)
              + 0.3 * f1_score(tr["label"], pred, zero_division=0))
        if val > best[0]:
            best = (val, lam)
    _, lam_i = best

    s_tr = lam_i * tr["deep_z"] + (1 - lam_i) * tr["classical_z"]
    tau_i = choose_threshold(tr["label"].to_numpy(), s_tr.to_numpy())

    te = blend_meta.loc[te_idx]
    s_te = lam_i * te["deep_z"] + (1 - lam_i) * te["classical_z"]
    sub = te[["event_id", "asset_id", "label"]].copy()
    sub["score"] = s_te.to_numpy()
    sub["pred"] = (sub["score"].to_numpy() >= tau_i).astype(int)
    sub["fold"] = i
    sub["lambda_deep_weight"] = lam_i
    sub["threshold"] = tau_i
    blend_rows.append(sub)

assert all(v == 0 for v in _leak_check), (
    "LEAK DETECTED: a test-fold event index appeared in its own fold's training subset "
    "during lambda/threshold selection. Do not trust any number below - stop and debug."
)
blended_oof = pd.concat(blend_rows, ignore_index=True)

assert len(blended_oof) == main_oof.event_id.nunique()
assert set(blended_oof.event_id) == set(main_oof.event_id.unique())
print("leak check: no test-fold event touched during its own fold's selection -> PASS")
print(f"blended OOF: {len(blended_oof)} events, covers every event exactly once (checked)")
print("lambda (deep weight) chosen per fold:",
      blended_oof.groupby("fold")["lambda_deep_weight"].first().round(3).to_dict())


# =====================================================================================
# STEP 6 - full, honest comparison table. Nothing is left out.
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 6 - FULL COMPARISON (58 pooled out-of-fold events, 27 positive)")
print("=" * 100)

comparison_rows = []
for nm in main_oof.model.unique():
    sub = main_oof[main_oof.model == nm]
    comparison_rows.append({"Model": nm, **pooled_metrics_df(sub),
                            **cluster_bootstrap_ci(sub)})

comparison_rows.append({"Model": f"Classical ({best_classical_col})",
                        **pooled_metrics_df(classical_oof_scored),
                        **cluster_bootstrap_ci(classical_oof_scored)})
comparison_rows.append({"Model": f"Deep ensemble ({'+'.join(ensemble_members)})",
                        **pooled_metrics_df(ensemble_oof),
                        **cluster_bootstrap_ci(ensemble_oof)})
comparison_rows.append({"Model": "Deep ensemble + classical BLEND (nested)",
                        **pooled_metrics_df(blended_oof),
                        **cluster_bootstrap_ci(blended_oof)})

comparison = pd.DataFrame(comparison_rows).sort_values("PR-AUC", ascending=False
                                                        ).reset_index(drop=True)
comparison.to_csv(RESULT / "V5_ensemble_blend_comparison.csv", index=False)

print(comparison[["Model", "PR-AUC", "PR-AUC CI low", "PR-AUC CI high",
                  "ROC-AUC", "Precision", "Recall", "F1", "MCC",
                  "TP", "FP", "TN", "FN"]].to_string(index=False))

blended_oof.to_csv(RESULT / "V5_blended_event_scores.csv", index=False)
ensemble_oof.to_csv(RESULT / "V5_ensemble_event_scores.csv", index=False)
print("\nsaved:", RESULT / "V5_ensemble_blend_comparison.csv")
print("saved:", RESULT / "V5_blended_event_scores.csv")
print("saved:", RESULT / "V5_ensemble_event_scores.csv")


# =====================================================================================
# STEP 7 - does the blend actually win, honestly, with significance context
# =====================================================================================

print("\n" + "=" * 100)
print("STEP 7 - did the blend actually help? (paired permutation test, asset-level)")
print("=" * 100)


def paired_permutation(oof_a, oof_b, metric="PR-AUC", n_perm=10000, seed=1):
    a = oof_a.sort_values("event_id").reset_index(drop=True)
    b = oof_b.sort_values("event_id").reset_index(drop=True)
    assert (a.event_id.to_numpy() == b.event_id.to_numpy()).all()
    fn = average_precision_score if metric == "PR-AUC" else roc_auc_score
    y = a.label.to_numpy(int)
    sa, sb = a.score.to_numpy(float), b.score.to_numpy(float)
    obs = fn(y, sa) - fn(y, sb)
    rng = np.random.default_rng(seed)
    assets = a.asset_id.unique()
    cnt = 0
    for _ in range(n_perm):
        flip = a.asset_id.map({t: bool(rng.integers(0, 2)) for t in assets}).to_numpy()
        pa = np.where(flip, sb, sa)
        pb = np.where(flip, sa, sb)
        if abs(fn(y, pa) - fn(y, pb)) >= abs(obs):
            cnt += 1
    return obs, (cnt + 1) / (n_perm + 1)


best_prior = comparison[comparison.Model != "Deep ensemble + classical BLEND (nested)"] \
    .iloc[comparison[comparison.Model != "Deep ensemble + classical BLEND (nested)"]
          ["PR-AUC"].values.argmax()]
print(f"best single component before blending: {best_prior.Model} "
      f"(PR-AUC {best_prior['PR-AUC']:.3f})")

blend_pr = float(comparison.loc[
    comparison.Model == "Deep ensemble + classical BLEND (nested)", "PR-AUC"].iloc[0])
print(f"blend PR-AUC: {blend_pr:.3f}")

ref_oof = (classical_oof_scored if best_prior.Model.startswith("Classical")
          else ensemble_oof if "ensemble" in best_prior.Model
          else main_oof[main_oof.model == best_prior.Model])
d, p = paired_permutation(blended_oof, ref_oof)
print(f"blend vs best prior component: dPR-AUC={d:+.3f}  p={p:.4f}")

if blend_pr > best_prior["PR-AUC"] and p < 0.05:
    print("\nRESULT: the blend improves on every individual component and the "
          "improvement clears p<0.05 in an asset-level paired permutation test.")
elif blend_pr > best_prior["PR-AUC"]:
    print("\nRESULT: the blend improves on every individual component numerically, "
          "but the improvement does NOT clear p<0.05. Report the point estimate "
          "alongside this p-value - do not claim significance.")
else:
    print("\nRESULT: the blend does NOT improve on the best individual component. "
          "This is itself a valid finding: report the classical baseline and the "
          "deep ensemble/blend side by side and let the numbers speak. Do not force "
          "the blend into the headline table if it did not help.")

print("\n" + "=" * 100)
print("STEP 7 COMPLETE - this cell adds a comparison table; it does not overwrite or")
print("hide CELL 22's rank statement, duration control, or classical-baseline result.")
print("=" * 100)